# How to Load Anything

This notebook demonstrates the unified Loader API in the TimeToAlign! library.
Each section exercises a loader following the same pattern:

1. **Loading**: `loader.load(*paths)` or `Loader.from_file(path)`
2. **Inspection**: `loader`, `loader.store`, `loader.store.<type_property>`
3. **Timeline creation**: `create_timeline()`, `create_timelines()`
4. **Event access**: `get_events()`, `get_event(id)`, `get_events_at(coord)`
5. **Timestamps**: `get_timestamp_at(coord)`, `get_timestamp_for(id)`,
   `get_timestamps_for(ids)`, `get_timestamp_table()`
6. **Groups** (where applicable): `create_group()`, group methods
7. **Bundles** (where applicable): `create_bundle()`, `create_match_claims()`

Retrieval names follow one grid, so the same four questions read the same way
on every object below. The suffix says what you are asking with: `_at` takes a
**position**, `_for` takes a **key** such as an event ID. The plural of the
noun takes many of them and gives back a list. The bare name
(`get_timestamp(...)`) is a convenience dispatcher that picks the precise
method from what you passed it. Tabulating is the same query with
`get_timestamp_table(...)`, which returns a PyArrow table by default and a
pandas frame with `format="dataframe"`.

## Setup

In [1]:
import warnings

from timetoalign import Provenance
from timetoalign.testdata import DATA_DIR, ensure_data
from timetoalign.timelines import Timeline

# How the example MatchClaims below were made
EXAMPLE_PROVENANCE = Provenance(method="manual", note="notebook example")

ensure_data("score", "midi", "vienna_1x22", "supra", "tabular", "thoresen")

SCORE_DIR = DATA_DIR / "score"
MIDI_DIR = DATA_DIR / "midi"
VIENNA_DIR = DATA_DIR / "vienna_1x22"

Every loader section below needs an event key to demonstrate the `_for`
getters and a short list of keys to demonstrate their plurals. Deriving those
is the same two lines each time, so they live here once.

In [2]:
def first_id(timeline: Timeline) -> str:
    """Return the ID of the timeline's first event."""
    return next(iter(timeline.get_events()))["id"]


def first_ids(timeline: Timeline, n: int = 5) -> list[str]:
    """Return the IDs of the timeline's first ``n`` events."""
    return [evt["id"] for evt in list(timeline.get_events())[:n]]

***
## 1. Ms3Loader (ms3-style score TSV)

Loads `.notes.tsv`, `.measures.tsv`, `.chords.tsv`, `.harmonies.tsv` files
produced by the ms3 library.  The `auto_discover=True` flag picks up companion
facets automatically.

In [3]:
from timetoalign import Ms3Loader

tsv_dir = SCORE_DIR / "flow_control" / "polyrythm_only"
tsv_notes = sorted(tsv_dir.glob("*polyrhythm_only.notes.tsv"))[0]
tsv_notes.name

'out_of_the_flow_experience-polyrhythm_only.notes.tsv'

In [4]:
tsv_loader = Ms3Loader(auto_discover=True)
tsv_loader.load(tsv_notes)
tsv_loader

Sources,"3 file(s): out_of_the_flow_experience-polyrhythm_only.notes.tsv, out_of_the_flow_experience-polyrhythm_only.measures.tsv, out_of_the_flow_experience-polyrhythm_only.chords.tsv"
Events,491
Unit,quarters
Try,"create_timeline(), get_events(...)"


In [5]:
tsv_loader.store

ScoreStore(notes=241, measures=14, controls=236, annotations=0)

In [6]:
tsv_loader.store.notes  # ScoreStore typed property -> NoteEventData

Events,241
Unit,quarters
Number type,fraction
Fields,midi : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [7]:
tsv_loader.store.notes.schema  # PyArrow schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
mc: int64
  -- field metadata --
  number_type: 'int64'
mn: string
mc_onset: struct<value: double, numerator: int64, denominator: int64>
  child 0, val

In [8]:
tsv_loader.store.measures  # MeasureData

Events,14
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [9]:
tsv_loader.store["notes"].to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,mc,mn,mc_onset,...,tpc,octave,tied,gracenote,chord_id,voice,staff,part_id,locator,parse_id
0,note_2ziptz8t6u70r,A♭1,interval,Note,0,3/20,3/20,1,0,0,...,-4,1,NaN,NaN,16,1,4,P1,row:0,parse_1nmvcw0ra670f
1,note_2jqca0f1n8742,C♭3,interval,Note,0,1/4,1/4,1,0,0,...,-7,3,NaN,NaN,12,1,3,P1,row:1,parse_1nmvcw0ra670f
2,note_3lh9pz8o39f6n,E♭3,interval,Note,0,1/4,1/4,1,0,0,...,-3,3,NaN,NaN,7,1,2,P1,row:2,parse_1nmvcw0ra670f
3,note_35r77uq70la3h,C♭6,interval,Note,0,3/14,3/14,1,0,0,...,-7,6,NaN,NaN,0,1,1,P1,row:3,parse_1nmvcw0ra670f
4,note_07jxwrko5r6tr,E♭2,interval,Note,3/20,3/10,3/20,1,0,3/80,...,-3,2,NaN,NaN,17,1,4,P1,row:4,parse_1nmvcw0ra670f


In [10]:
with warnings.catch_warnings(record=True) as tsv_warnings:
    warnings.simplefilter("always")
    tsv_tl = tsv_loader.create_timeline()
tsv_tl

ContinuousLogicalTimeline(id='clt_09d8r2lvhnyx4', length=93/2, unit=quarters, events=0, children=2, cmaps=1)

The measures of this polyrhythm fragment do not add up to their bars, and
building the timeline warns which. The cell above recorded the warning
instead of letting it print:

In [11]:
print(tsv_warnings[0].message)

Floating measures that do not add up to their bars:
  short: measures '3', '4', '5', '6' (number 2) end 5 quarters into a bar of 6, and measure '7' (number 3), which they lead to, does not complete the bar;
  short: measure '7' (number 3) ends 2 quarters into a bar of 6, and measure '8' (number 4), which it leads to, does not complete the bar;
  short: measures '12', '13' (number 8) end 4 quarters into a bar of 6, and measure '14' (number 9), which they lead to, does not complete the bar


In [12]:
tsv_tl.get_events()

Events,477
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [13]:
tsv_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
chord_id: string
gracenote: string
mc: int64
  -- field metadata --
  number_type: 'int64'
mc_onset: struct<value: double, numerator: int64, denominato

In [14]:
tsv_tl.get_events(temporal_type="interval", min_coord=0, max_coord=10)

Events,114
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [15]:
tsv_tl.get_events(event_type="Note", min_coord=0, max_coord=10)

Events,57
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [16]:
first_event_id = first_id(tsv_tl)
first_event_id

'note_2ziptz8t6u70r'

In [17]:
tsv_tl.get_event(first_event_id)

{'id': 'note_2ziptz8t6u70r',
 'name': 'A♭1',
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 0.15, 'numerator': 3, 'denominator': 20},
 'duration': {'value': 0.15, 'numerator': 3, 'denominator': 20},
 'mc': 1,
 'mn': '0',
 'mc_onset': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'mn_onset': {'value': 1.25, 'numerator': 5, 'denominator': 4},
 'specific_pitch': {'step': 'A', 'alter': -1, 'octave': 1, 'cents': 0.0},
 'midi': 32,
 'tpc': -4,
 'octave': 1,
 'tied': None,
 'gracenote': None,
 'chord_id': '16',
 'voice': 1,
 'staff': 4,
 'part_id': 'P1',
 'source_timeline': 'clt_1wsz6ewxdhqsr'}

In [18]:
tsv_tl.get_timestamp_for(first_event_id)

TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(3, 20), quarters), source='clt_09d8r2lvhnyx4')

In [19]:
tsv_tl.get_timestamp_at(5)

ID,Coordinate,Type
clt_09d8r2lvhnyx4,5 quarters,axis
clt_1wsz6ewxdhqsr,5 quarters,child
clt_35c7vydv2s8o2,5 quarters,child
floating_measures,1.666 floating_measures,cmap


In [20]:
tsv_events_at = tsv_tl.get_events_at(5)
tsv_events_at

{'clt_1wsz6ewxdhqsr': [{'id': 'note_1j3buli7ehnrm',
   'name': 'A♭3',
   'temporal_type': 'interval',
   'event_type': 'Note',
   'start': Fraction(1, 1),
   'end': Fraction(7, 1),
   'duration': Fraction(6, 1),
   'mc': 2,
   'mn': '1',
   'mc_onset': Fraction(0, 1),
   'mn_onset': Fraction(0, 1),
   'specific_pitch': {'step': 'A', 'alter': -1, 'octave': 3, 'cents': 0.0},
   'midi': 56,
   'tpc': -4,
   'octave': 3,
   'tied': None,
   'gracenote': None,
   'chord_id': '21',
   'voice': 1,
   'staff': 1,
   'part_id': 'P1'},
  {'id': 'note_1rpt1n92ayqd6',
   'name': 'C5',
   'temporal_type': 'interval',
   'event_type': 'Note',
   'start': Fraction(4, 1),
   'end': Fraction(6, 1),
   'duration': Fraction(2, 1),
   'mc': 2,
   'mn': '1',
   'mc_onset': Fraction(3, 4),
   'mn_onset': Fraction(3, 4),
   'specific_pitch': {'step': 'C', 'alter': 0, 'octave': 5, 'cents': 0.0},
   'midi': 72,
   'tpc': 0,
   'octave': 5,
   'tied': 1,
   'gracenote': None,
   'chord_id': '23',
   'voice': 1,

In [21]:
tsv_tl.get_timestamp_table(format="dataframe")

,clt_09d8r2lvhnyx4 (quarters),clt_1wsz6ewxdhqsr (quarters),clt_35c7vydv2s8o2 (quarters),quarters_to_measures (floating_measures)
id,,,,
note_2ziptz8t6u70r,0,0,0,0.833
note_07jxwrko5r6tr,3/20,3/20,3/20,0.858
note_1kvh9rf53n3e9,3/14,3/14,3/14,0.869
note_3469qf8chdym8,1/4,1/4,1/4,0.875
note_1q0h1ps45c247,3/10,3/10,3/10,0.883
...,...,...,...,...
note_298yt05qavi9w,635/14,635/14,635/14,10.119
note_0y8lkmmncehi6,91/2,91/2,91/2,10.166
,183/4,183/4,183/4,10.250


In [22]:
tsv_event_ids = first_ids(tsv_tl)
tsv_tl.get_timestamps_for(tsv_event_ids)

[TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(3, 20), quarters), source='clt_09d8r2lvhnyx4'),
 TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 4), quarters), source='clt_09d8r2lvhnyx4'),
 TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 4), quarters), source='clt_09d8r2lvhnyx4'),
 TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(3, 14), quarters), source='clt_09d8r2lvhnyx4'),
 TimeIntervalStamp(start=Coordinate(Fraction(3, 20), quarters), end=Coordinate(Fraction(3, 10), quarters), source='clt_09d8r2lvhnyx4')]

In [23]:
tsv_tl.get_timestamp_table()

pyarrow.Table
clt_09d8r2lvhnyx4: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_1wsz6ewxdhqsr: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_35c7vydv2s8o2: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
quarters_to_measures: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
clt_09d8r2lvhnyx4: [
  -- is_valid: all not null
  -- child 0 type: double
[0,0.15,0.21428571428571427,0.25,0.3,...,45.357142857142854,45.5,45.75,45.92857142857143,46.5]
  -- child 1 type: int64
[0,3,3,1,3,...,635,91,183,643,93]
  -- child 2 type: int64
[1,20,14,4,10,...,14,2,4,14,2]]
clt_1wsz6ewxdhqsr: [
  -- is_valid: all not 

***
## 2. PartituraLoader (MusicXML via partitura)

Parses MusicXML (and MIDI) files using the `partitura` library.

In [24]:
from timetoalign.loader.score.partitura import PartituraLoader

partitura_file = (
    SCORE_DIR / "flow_control" / "out_of_the_flow_experience-flow_only.musicxml"
)
partitura_file.name

'out_of_the_flow_experience-flow_only.musicxml'

In [25]:
partitura_loader = PartituraLoader()
partitura_loader.load(partitura_file)
partitura_loader

Sources,1 file(s): out_of_the_flow_experience-flow_only.musicxml
Events,81
Unit,quarters
Try,"create_timeline(), get_events(...)"


In [26]:
partitura_loader.store

ScoreStore(notes=37, measures=15, controls=25, annotations=4)

In [27]:
partitura_loader.store.notes

Events,37
Unit,quarters
Number type,fraction
Fields,midi : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [28]:
partitura_loader.store.notes.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
mc: int64
  -- field metadata --
  number_type: 'int64'
mn: string
mc_onset: struct<value: double, numerator: int64, denominator: int64>
  child 0, val

In [29]:
partitura_loader.store.measures

Events,15
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [30]:
partitura_loader.store["notes"].to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,mc,mn,mc_onset,...,tpc,octave,tied,gracenote,chord_id,voice,staff,part_id,locator,parse_id
0,note_0iq3kmiuajay7,A♭2,interval,Note,0,1,1,1,1,0,...,-4,2,NaN,NaN,NaN,1,1,P1,part:0 element:6,parse_2gmpcu3bb66kk
1,note_1zcc4je480i2k,B♭2,interval,Note,1,2,1,2,2,0,...,-2,2,NaN,NaN,NaN,1,1,P1,part:0 element:9,parse_2gmpcu3bb66kk
2,note_0qgif1guwt2vh,C♭3,interval,Note,2,3,1,2,2,1,...,-7,3,NaN,NaN,NaN,1,1,P1,part:0 element:10,parse_2gmpcu3bb66kk
3,note_3lxdgygvnddox,D♭3,interval,Note,3,4,1,2,2,2,...,-5,3,NaN,NaN,NaN,1,1,P1,part:0 element:11,parse_2gmpcu3bb66kk
4,note_2efznfevc3xcn,E♭3,interval,Note,4,5,1,2,2,3,...,-3,3,NaN,NaN,NaN,1,1,P1,part:0 element:12,parse_2gmpcu3bb66kk


In [31]:
partitura_tl = partitura_loader.create_timeline()
partitura_tl

ContinuousLogicalTimeline(id='clt_3f30jypf0ya99', length=77/2, unit=quarters, events=0, children=4, cmaps=2)

In [32]:
partitura_tl.get_events()

Events,81
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [33]:
partitura_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
actual_length: double
  -- field metadata --
  description: 'Real duration (may differ)'
barline: string
breaks: string
  -- field metadata --
  descri

In [34]:
partitura_tl.get_events(temporal_type="interval", min_coord=0, max_coord=20)

Events,28
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [35]:
part_first_event_id = first_id(partitura_tl)
partitura_tl.get_event(part_first_event_id)

{'id': 'note_0iq3kmiuajay7',
 'name': 'A♭2',
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'duration': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'mc': 1,
 'mn': '1',
 'mc_onset': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'mn_onset': None,
 'specific_pitch': {'step': 'A', 'alter': -1, 'octave': 2, 'cents': 0.0},
 'midi': 44,
 'tpc': -4,
 'octave': 2,
 'tied': None,
 'gracenote': None,
 'chord_id': None,
 'voice': 1,
 'staff': 1,
 'part_id': 'P1',
 'source_timeline': 'clt_05cda074jd3o2'}

In [36]:
partitura_tl.get_timestamp_for(part_first_event_id)

TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 1), quarters), source='clt_3f30jypf0ya99')

In [37]:
partitura_tl.get_timestamp_at(10)

ID,Coordinate,Type
clt_3f30jypf0ya99,10 quarters,axis
clt_05cda074jd3o2,10 quarters,child
clt_090u2f4wph0am,10 quarters,child
clt_19vib7s2qe8d9,10 quarters,child
clt_0vx6m025cuowe,10 quarters,child
floating_measures,6.666 floating_measures,cmap
quarters,9 quarters,cmap


In [38]:
partitura_tl.get_events_at(0)

{'clt_05cda074jd3o2': [{'id': 'note_0iq3kmiuajay7',
   'name': 'A♭2',
   'temporal_type': 'interval',
   'event_type': 'Note',
   'start': Fraction(0, 1),
   'end': Fraction(1, 1),
   'duration': Fraction(1, 1),
   'mc': 1,
   'mn': '1',
   'mc_onset': Fraction(0, 1),
   'mn_onset': None,
   'specific_pitch': {'step': 'A', 'alter': -1, 'octave': 2, 'cents': 0.0},
   'midi': 44,
   'tpc': -4,
   'octave': 2,
   'tied': None,
   'gracenote': None,
   'chord_id': None,
   'voice': 1,
   'staff': 1,
   'part_id': 'P1'}],
 'clt_090u2f4wph0am': [{'id': 'measure_3lgb7gqniteed',
   'name': '1',
   'temporal_type': 'interval',
   'event_type': 'Measure',
   'start': Fraction(0, 1),
   'end': Fraction(1, 1),
   'duration': Fraction(1, 1),
   'mc': 1,
   'mn': '1',
   'mn_int': None,
   'mm_id': None,
   'nominal_length': None,
   'actual_length': None,
   'mc_offset': None,
   'quarterbeats_all_endings': None,
   'timesig': None,
   'timesig_num': None,
   'timesig_den': None,
   'keysig': None,

In [39]:
partitura_tl.get_timestamp_table(format="dataframe")

,clt_3f30jypf0ya99 (quarters),clt_05cda074jd3o2 (quarters),clt_090u2f4wph0am (quarters),clt_19vib7s2qe8d9 (quarters),clt_0vx6m025cuowe (quarters),quarters_to_measures (floating_measures),raw_quarters (quarters)
id,,,,,,,
note_0iq3kmiuajay7,0,0,0,0,0,1.000,-1
note_1zcc4je480i2k,1,1,1,1,1,2.000,0
note_0qgif1guwt2vh,2,2,2,2,2,2.250,1
note_3lxdgygvnddox,3,3,3,3,3,2.500,2
note_2efznfevc3xcn,4,4,4,4,4,2.750,3
note_3ulhns67wltpg,5,5,5,5,5,3.000,4
note_1vqbehdw2sl1p,6,6,6,6,6,4.000,5
note_2t6c6fq81w9y1,7,7,7,7,7,5.000,6
note_2mtc9ue1hmh9s,8,8,8,8,8,5.500,7


In [40]:
part_event_ids = first_ids(partitura_tl)
partitura_tl.get_timestamps_for(part_event_ids)

[TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 1), quarters), source='clt_3f30jypf0ya99'),
 TimeIntervalStamp(start=Coordinate(Fraction(1, 1), quarters), end=Coordinate(Fraction(2, 1), quarters), source='clt_3f30jypf0ya99'),
 TimeIntervalStamp(start=Coordinate(Fraction(2, 1), quarters), end=Coordinate(Fraction(3, 1), quarters), source='clt_3f30jypf0ya99'),
 TimeIntervalStamp(start=Coordinate(Fraction(3, 1), quarters), end=Coordinate(Fraction(4, 1), quarters), source='clt_3f30jypf0ya99'),
 TimeIntervalStamp(start=Coordinate(Fraction(4, 1), quarters), end=Coordinate(Fraction(5, 1), quarters), source='clt_3f30jypf0ya99')]

In [41]:
partitura_tl.get_timestamp_table()

pyarrow.Table
clt_3f30jypf0ya99: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_05cda074jd3o2: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_090u2f4wph0am: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_19vib7s2qe8d9: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_0vx6m025cuowe: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
quarters_to_measures: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
raw_quarter

***
## 3. Music21Loader (MusicXML/MEI via music21)

Parses MusicXML and MEI files using the `music21` library. music21 gives a
measure without timed content no length, so a source holding such measures
(the MEI skeleton of this piece, whose measures are empty) states no measure
positions through it and the load raises `ValueError`. The piece's MusicXML
times every measure.

In [42]:
from timetoalign.loader.score.music21 import Music21Loader

# Use a small MusicXML file
music21_file = (
    SCORE_DIR
    / "beethoven_op18-4iv_multimodal"
    / "op18_no4_mov4_flow"
    / "op18_no4_mov4_flow.musicxml"
)
music21_file.name

'op18_no4_mov4_flow.musicxml'

In [43]:
music21_loader = Music21Loader()
music21_loader.load(music21_file)
music21_loader

Sources,1 file(s): op18_no4_mov4_flow.musicxml
Events,231
Unit,quarters
Try,"create_timeline(), get_events(...)"


In [44]:
music21_loader.store

ScoreStore(notes=0, measures=226, controls=5, annotations=0)

In [45]:
music21_loader.store.notes

Events,0
Unit,quarters
Number type,fraction
Fields,midi : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [46]:
music21_loader.store.notes.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
mc: int64
  -- field metadata --
  number_type: 'int64'
mn: string
mc_onset: struct<value: double, numerator: int64, denominator: int64>
  child 0, val

In [47]:
music21_loader.store.measures

Events,226
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [48]:
music21_tl = music21_loader.create_timeline()
music21_tl

ContinuousLogicalTimeline(id='clt_390a0tfssvp2t', length=880, unit=quarters, events=0, children=2, cmaps=1)

In [49]:
music21_tl.get_events()

Events,231
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [50]:
music21_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
actual_length: double
  -- field metadata --
  description: 'Real duration (may differ)'
barline: string
breaks: string
  -- field metadata --
  descri

In [51]:
music21_tl.get_events(temporal_type="interval", min_coord=0, max_coord=20)

Events,6
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [52]:
m21_first_event_id = first_id(music21_tl)
music21_tl.get_event(m21_first_event_id)

{'id': 'measure_3h4t4wip6yude',
 'name': '1',
 'temporal_type': 'interval',
 'event_type': 'Measure',
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'duration': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'mc': 1,
 'mn': '1',
 'mn_int': None,
 'mm_id': None,
 'nominal_length': None,
 'actual_length': None,
 'mc_offset': None,
 'quarterbeats_all_endings': None,
 'timesig': None,
 'timesig_num': None,
 'timesig_den': None,
 'keysig': None,
 'keysig_fifths': None,
 'keysig_mode': None,
 'start_repeat': False,
 'end_repeat': False,
 'next': None,
 'volta': None,
 'volta_list': None,
 'repeats': None,
 'breaks': None,
 'markers': None,
 'jump_bwd': None,
 'jump_fwd': None,
 'play_until': None,
 'dont_count': None,
 'numbering_offset': None,
 'barline': None,
 'part_id': 'Violoncello',
 'source_timeline': 'clt_10m5gnny1in3m'}

In [53]:
music21_tl.get_timestamp_for(m21_first_event_id)

TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 1), quarters), source='clt_390a0tfssvp2t')

In [54]:
music21_tl.get_timestamp_at(10)

ID,Coordinate,Type
clt_390a0tfssvp2t,10 quarters,axis
clt_10m5gnny1in3m,10 quarters,child
clt_0dq9v2tbriwtl,10 quarters,child
floating_measures,4.25 floating_measures,cmap


In [55]:
music21_tl.get_events_at(0)

{'clt_10m5gnny1in3m': [{'id': 'measure_3h4t4wip6yude',
   'name': '1',
   'temporal_type': 'interval',
   'event_type': 'Measure',
   'start': Fraction(0, 1),
   'end': Fraction(1, 1),
   'duration': Fraction(1, 1),
   'mc': 1,
   'mn': '1',
   'mn_int': None,
   'mm_id': None,
   'nominal_length': None,
   'actual_length': None,
   'mc_offset': None,
   'quarterbeats_all_endings': None,
   'timesig': None,
   'timesig_num': None,
   'timesig_den': None,
   'keysig': None,
   'keysig_fifths': None,
   'keysig_mode': None,
   'start_repeat': False,
   'end_repeat': False,
   'next': None,
   'volta': None,
   'volta_list': None,
   'repeats': None,
   'breaks': None,
   'markers': None,
   'jump_bwd': None,
   'jump_fwd': None,
   'play_until': None,
   'dont_count': None,
   'numbering_offset': None,
   'barline': None,
   'part_id': 'Violoncello'}],
 'clt_0dq9v2tbriwtl': [{'id': 'metronomemark_2dog5jd5ji6tx',
   'name': 'MetronomeMark',
   'temporal_type': 'instant',
   'event_type': 

In [56]:
music21_tl.get_timestamp_table(format="dataframe")

,clt_390a0tfssvp2t (quarters),clt_10m5gnny1in3m (quarters),clt_0dq9v2tbriwtl (quarters),quarters_to_measures (floating_measures)
id,,,,
measure_3h4t4wip6yude,0,0,0,1.0
measure_04fnkkamia06b,1,1,1,2.0
measure_38fb68uk9uarc,5,5,5,3.0
measure_1m01y8upazj66,9,9,9,4.0
measure_0p3v9ewbls2md,13,13,13,5.0
...,...,...,...,...
measure_1ve7sqf63zgbi,865,865,None,223.0
measure_1kaglq7gj6poa,869,869,None,224.0
measure_264m3wznj0ljf,873,873,None,225.0


In [57]:
m21_event_ids = first_ids(music21_tl)
music21_tl.get_timestamps_for(m21_event_ids)

[TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 1), quarters), source='clt_390a0tfssvp2t'),
 TimeIntervalStamp(start=Coordinate(Fraction(1, 1), quarters), end=Coordinate(Fraction(5, 1), quarters), source='clt_390a0tfssvp2t'),
 TimeIntervalStamp(start=Coordinate(Fraction(5, 1), quarters), end=Coordinate(Fraction(9, 1), quarters), source='clt_390a0tfssvp2t'),
 TimeIntervalStamp(start=Coordinate(Fraction(9, 1), quarters), end=Coordinate(Fraction(13, 1), quarters), source='clt_390a0tfssvp2t'),
 TimeIntervalStamp(start=Coordinate(Fraction(13, 1), quarters), end=Coordinate(Fraction(17, 1), quarters), source='clt_390a0tfssvp2t')]

In [58]:
music21_tl.get_timestamp_table()

pyarrow.Table
clt_390a0tfssvp2t: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_10m5gnny1in3m: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
clt_0dq9v2tbriwtl: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
quarters_to_measures: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
clt_390a0tfssvp2t: [
  -- is_valid: all not null
  -- child 0 type: double
[0,1,5,9,13,...,865,869,873,877,880]
  -- child 1 type: int64
[0,1,5,9,13,...,865,869,873,877,880]
  -- child 2 type: int64
[1,1,1,1,1,...,1,1,1,1,1]]
clt_10m5gnny1in3m: [
  -- is_valid: all not null
  -- child 0 type: double
[0,1,5,9,13,...,865,869,873,8

***
## 4. MeasureMapLoader (.mm.json)

Parses MeasureMap JSON files describing measure boundaries, time signatures,
and flow control.

In [59]:
from timetoalign.loader.score.measuremap import MeasureMapLoader

mm_file = (
    SCORE_DIR
    / "beethoven_op18-4iv_multimodal"
    / "ABC"
    / "n04op18-4_04.measures.mm.json"
)
mm_file.name

'n04op18-4_04.measures.mm.json'

In [60]:
mm_loader = MeasureMapLoader()
mm_loader.load(mm_file)
mm_loader

Sources,1 file(s): n04op18-4_04.measures.mm.json
Events,226
Unit,quarters
Try,"create_timeline(), get_events(...)"


In [61]:
mm_loader.store

ScoreStore(notes=0, measures=226, controls=0, annotations=0)

In [62]:
mm_loader.store.measures

Events,226
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [63]:
mm_loader.store["measures"].to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,mc,mn,mn_int,...,markers,jump_bwd,jump_fwd,play_until,dont_count,numbering_offset,barline,part_id,locator,parse_id
0,measure_1iyq670ahjza1,M0,interval,Measure,0,1,1,1,0,0,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,measure:1 key:1,parse_1jpt8gzjfq8fm
1,measure_21ul9j8g5cmky,M1,interval,Measure,1,5,4,2,1,1,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,measure:2 key:2,parse_1jpt8gzjfq8fm
2,measure_02hqhbjkc1prk,M2,interval,Measure,5,9,4,3,2,2,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,measure:3 key:3,parse_1jpt8gzjfq8fm
3,measure_24d1bdo3u6uo9,M3,interval,Measure,9,13,4,4,3,3,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,measure:4 key:4,parse_1jpt8gzjfq8fm
4,measure_0dscsrh3bdvmb,M4,interval,Measure,13,17,4,5,4,4,...,NaN,NaN,NaN,NaN,None,NaN,NaN,NaN,measure:5 key:5,parse_1jpt8gzjfq8fm


In [64]:
with warnings.catch_warnings(record=True) as mm_warnings:
    warnings.simplefilter("always")
    mm_tl = mm_loader.create_timeline()
mm_tl

ContinuousLogicalTimeline(id='clt_3p166no4wgxqr', length=1757/2, unit=quarters, events=226, children=0, cmaps=1)

Building the timeline warns where a measure, joined with the next that
shares its number, runs past its bar; recorded, the warning reads:

In [65]:
print(mm_warnings[0].message)

Floating measures that do not add up to their bars:
  overlength: measures '44', '45' (number 40) reach 7 quarters into a bar of 4;
  overlength: measures '93', '94' (number 86) reach 15/2 quarters into a bar of 4;
  overlength: measures '102', '103' (number 94) reach 8 quarters into a bar of 4


In [66]:
mm_tl.get_events()

Events,226
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


The loader writes the repeat signs the measure map states as the timeline's
flow control; `get_flow()` plays them and returns the played length and what
playing found.

In [67]:
mm_tl.flow_control

FlowControlRegistry(7 jumps, 7 jump_targets, 0 breaks, 0 section_breaks, 0 barlines)

In [68]:
mm_flow = mm_tl.get_flow()
{"played length": mm_flow.length, "findings": mm_flow.findings}

{'played length': Coordinate(Fraction(2253, 2), quarters), 'findings': ()}

***
## 5. PerformanceMidiLoader (performance MIDI via mido)

Low-level MIDI parsing using `mido`.  Pairs note_on/note_off events,
handles running status, and extracts control changes.

In [69]:
from timetoalign.loader.midi.performance import PerformanceMidiLoader

perf_midi_file = MIDI_DIR / "performance" / "rachmaninoff_perf.mid"
perf_midi_file.name

'rachmaninoff_perf.mid'

In [70]:
perf_midi_loader = PerformanceMidiLoader()
perf_midi_loader.load(perf_midi_file)
perf_midi_loader

Sources,1 file(s): rachmaninoff_perf.mid
Events,111
Unit,ticks
Try,"create_timeline(), get_events(...)"


In [71]:
perf_midi_loader.store

MidiStore(notes=111, controls=0)

In [72]:
perf_midi_loader.store.notes  # MidiStore typed property -> MidiEventData

Events,111
Unit,ticks
Number type,int
Fields,pitch : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [73]:
perf_midi_loader.store.notes.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
pitch: int64
velocity: int8
channel: int8
track: int16
control: int8
value: int8
program: int8
locator: string
parse_id: string
-- schema metadata --
timetoalign: '{"loader_cl

In [74]:
perf_midi_loader.store["notes"].to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,pitch,velocity,channel,track,control,value,program,locator,parse_id
0,note_0oety7ky5m1jk,NaN,interval,Note,91,189,98,24,127,0,0,NaN,NaN,NaN,track:0 message:0,parse_1k9762uwsdqn6
1,note_3mti6si6sa2wk,NaN,interval,Note,189,287,98,36,127,0,0,NaN,NaN,NaN,track:0 message:2,parse_1k9762uwsdqn6
2,note_2k1ovhvf7kyv4,NaN,interval,Note,287,403,116,43,127,0,0,NaN,NaN,NaN,track:0 message:4,parse_1k9762uwsdqn6
3,note_1awnwghjcbt02,NaN,interval,Note,412,527,115,48,127,0,0,NaN,NaN,NaN,track:0 message:6,parse_1k9762uwsdqn6
4,note_01o4ndzmq7iqq,NaN,interval,Note,528,626,98,51,127,0,0,NaN,NaN,NaN,track:0 message:8,parse_1k9762uwsdqn6


In [75]:
perf_midi_tl = perf_midi_loader.create_timeline()
perf_midi_tl

DiscreteLogicalTimeline(id='dlt_09ypmniiftb8u', length=11537, unit=ticks, events=111, children=0, cmaps=2)

In [76]:
perf_midi_tl.get_events()

Events,111
Unit,ticks
Number type,int
Fields,pitch : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [77]:
perf_midi_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
pitch: int64
velocity: int8
channel: int8
track: int16
control: int8
value: int8
program: int8
-- schema metadata --
timetoalign: '{"loader_class": "MidiEventData", "number_ty

In [78]:
perf_midi_tl.get_events(temporal_type="interval")

Events,111
Unit,ticks
Number type,int
Fields,pitch : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [79]:
perf_midi_first_event_id = first_id(perf_midi_tl)
perf_midi_tl.get_event(perf_midi_first_event_id)

{'id': 'note_0oety7ky5m1jk',
 'name': None,
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 91.0, 'numerator': 91, 'denominator': 1},
 'end': {'value': 189.0, 'numerator': 189, 'denominator': 1},
 'duration': {'value': 98.0, 'numerator': 98, 'denominator': 1},
 'pitch': 24,
 'velocity': 127,
 'channel': 0,
 'track': 0,
 'control': None,
 'value': None,
 'program': None}

In [80]:
perf_midi_tl.get_timestamp_for(perf_midi_first_event_id)

TimeIntervalStamp(start=Coordinate(91, ticks), end=Coordinate(189, ticks), source='dlt_09ypmniiftb8u')

In [81]:
perf_midi_tl.get_timestamp_at(0.5)

ID,Coordinate,Type
dlt_09ypmniiftb8u,0 ticks,axis
quarters,0 quarters,cmap
seconds,0 seconds,cmap


In [82]:
perf_midi_tl.get_events_at(0)

{}

In [83]:
perf_midi_tl.get_timestamp_table(format="dataframe")

,dlt_09ypmniiftb8u (ticks),ticks_to_quarters (quarters),ticks_to_seconds (seconds)
id,,,
note_0oety7ky5m1jk,91,1707614858711313/9007199254740992,0.094792
note_3mti6si6sa2wk,189,7093169413108531/18014398509481984,0.196875
note_2k1ovhvf7kyv4,287,2692777277198609/4503599627370496,0.298958
,403,3781147187146479/4503599627370496,0.419792
note_1awnwghjcbt02,412,7731179360319351/9007199254740992,0.429167
...,...,...,...
note_1mn0k4cqdkcnu,11291,3310556210458351/140737488355328,11.761458
,11394,6681512259669197/281474976710656,11.868750
note_04yyfz6kn4lgh,11395,6682098665870677/281474976710656,11.869792


In [84]:
perf_midi_event_ids = first_ids(perf_midi_tl)
perf_midi_tl.get_timestamps_for(perf_midi_event_ids)

[TimeIntervalStamp(start=Coordinate(91, ticks), end=Coordinate(189, ticks), source='dlt_09ypmniiftb8u'),
 TimeIntervalStamp(start=Coordinate(189, ticks), end=Coordinate(287, ticks), source='dlt_09ypmniiftb8u'),
 TimeIntervalStamp(start=Coordinate(287, ticks), end=Coordinate(403, ticks), source='dlt_09ypmniiftb8u'),
 TimeIntervalStamp(start=Coordinate(412, ticks), end=Coordinate(527, ticks), source='dlt_09ypmniiftb8u'),
 TimeIntervalStamp(start=Coordinate(528, ticks), end=Coordinate(626, ticks), source='dlt_09ypmniiftb8u')]

In [85]:
perf_midi_tl.get_timestamp_table()

pyarrow.Table
dlt_09ypmniiftb8u: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
ticks_to_quarters: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
ticks_to_seconds: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
dlt_09ypmniiftb8u: [
  -- is_valid: all not null
  -- child 0 type: double
[91,189,287,403,412,...,11291,11394,11395,11487,11537]
  -- child 1 type: int64
[91,189,287,403,412,...,11291,11394,11395,11487,11537]
  -- child 2 type: int64
[1,1,1,1,1,...,1,1,1,1,1]]
ticks_to_quarters: [
  -- is_valid: all not null
  -- child 0 type: double
[0.18958333333333333,0.39375,0.5979166666666667,0.8395833333333333,0.8583333333333333,...,23.522916666666667,23.7375,23.739583333333332,23.93125,24.03541666666

***
## 6. ScoreMidiLoader (score MIDI via partitura)

Parses score-like MIDI files using `partitura` for structural information
(parts, voices, time signatures).

In [86]:
from timetoalign.loader.midi.score import ScoreMidiLoader

score_midi_file = MIDI_DIR / "score" / "beethoven_mtd.mid"
score_midi_file.name

'beethoven_mtd.mid'

In [87]:
score_midi_loader = ScoreMidiLoader()
score_midi_loader.load(score_midi_file)
score_midi_loader

Sources,1 file(s): beethoven_mtd.mid
Events,16
Unit,ticks
Try,"create_timeline(), get_events(...)"


In [88]:
score_midi_loader.store

MidiStore(notes=16, controls=0)

In [89]:
score_midi_loader.store.notes

Events,16
Unit,ticks
Number type,int
Fields,pitch : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [90]:
score_midi_loader.store.notes.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
pitch: int64
velocity: int8
channel: int8
track: int16
control: int8
value: int8
program: int8
voice: int8
staff: int8
part_id: string
locator: string
parse_id: string
-- sche

In [91]:
score_midi_loader.store["notes"].to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,pitch,velocity,channel,track,control,value,program,voice,staff,part_id,locator,parse_id
0,note_3t9t5jbn352dt,NaN,interval,Note,0,239,239,34,80,NaN,NaN,NaN,NaN,NaN,1,0,NaN,track:0 message:9,parse_3uq3lc4ivcfzz
1,note_39x9jy3m93kjr,NaN,interval,Note,240,959,719,62,80,NaN,NaN,NaN,NaN,NaN,1,0,NaN,track:0 message:12,parse_3uq3lc4ivcfzz
2,note_1zmom8i4iyxjm,NaN,interval,Note,960,1199,239,86,80,NaN,NaN,NaN,NaN,NaN,1,0,NaN,track:0 message:14,parse_3uq3lc4ivcfzz
3,note_3fblu1xn68mk2,NaN,interval,Note,1200,1679,479,86,80,NaN,NaN,NaN,NaN,NaN,1,0,NaN,track:0 message:16,parse_3uq3lc4ivcfzz
4,note_0vb3k7f0hg8fo,NaN,interval,Note,1680,1919,239,87,80,NaN,NaN,NaN,NaN,NaN,1,0,NaN,track:0 message:18,parse_3uq3lc4ivcfzz


In [92]:
score_midi_tl = score_midi_loader.create_timeline()
score_midi_tl

DiscreteLogicalTimeline(id='dlt_2yklvx93ukbwa', length=6960, unit=ticks, events=16, children=0, cmaps=2)

In [93]:
score_midi_tl.get_events()

Events,16
Unit,ticks
Number type,int
Fields,pitch : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [94]:
score_midi_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "int", "unit": "ticks", "version": 1}'
pitch: int64
velocity: int8
channel: int8
track: int16
control: int8
value: int8
program: int8
voice: int8
staff: int8
part_id: string
-- schema metadata --
timetoalign: '{"lo

In [95]:
score_midi_tl.get_events(temporal_type="interval", min_coord=0, max_coord=10)

Events,1
Unit,ticks
Number type,int
Fields,pitch : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [96]:
score_midi_first_event_id = first_id(score_midi_tl)
score_midi_tl.get_event(score_midi_first_event_id)

{'id': 'note_3t9t5jbn352dt',
 'name': None,
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 239.0, 'numerator': 239, 'denominator': 1},
 'duration': {'value': 239.0, 'numerator': 239, 'denominator': 1},
 'pitch': 34,
 'velocity': 80,
 'channel': None,
 'track': None,
 'control': None,
 'value': None,
 'program': None,
 'voice': 1,
 'staff': 0,
 'part_id': None}

In [97]:
score_midi_tl.get_timestamp_for(score_midi_first_event_id)

TimeIntervalStamp(start=Coordinate(0, ticks), end=Coordinate(239, ticks), source='dlt_2yklvx93ukbwa')

In [98]:
score_midi_tl.get_timestamp_at(0)

ID,Coordinate,Type
dlt_2yklvx93ukbwa,0 ticks,axis
quarters,0 quarters,cmap
seconds,0 seconds,cmap


In [99]:
score_midi_tl.get_events_at(0)

{'dlt_2yklvx93ukbwa': [{'id': 'note_3t9t5jbn352dt',
   'name': None,
   'temporal_type': 'interval',
   'event_type': 'Note',
   'start': Fraction(0, 1),
   'end': Fraction(239, 1),
   'duration': Fraction(239, 1),
   'pitch': 34,
   'velocity': 80,
   'channel': None,
   'track': None,
   'control': None,
   'value': None,
   'program': None,
   'voice': 1,
   'staff': 0,
   'part_id': None}]}

In [100]:
score_midi_tl.get_timestamp_table(format="dataframe")

,dlt_2yklvx93ukbwa (ticks),ticks_to_quarters (quarters),ticks_to_seconds (seconds)
id,,,
note_3t9t5jbn352dt,0,0,0.000000
,239,4484834628923119/9007199254740992,0.248958
note_39x9jy3m93kjr,240,1/2,0.250000
,959,8997816755517303/4503599627370496,0.998958
note_1zmom8i4iyxjm,960,2,1.000000
,1199,1406202071150319/562949953421312,1.248958
note_3fblu1xn68mk2,1200,5/2,1.250000
,1679,1969152024571631/562949953421312,1.748958
note_0vb3k7f0hg8fo,1680,7/2,1.750000


In [101]:
score_midi_event_ids = first_ids(score_midi_tl)
score_midi_tl.get_timestamps_for(score_midi_event_ids)

[TimeIntervalStamp(start=Coordinate(0, ticks), end=Coordinate(239, ticks), source='dlt_2yklvx93ukbwa'),
 TimeIntervalStamp(start=Coordinate(240, ticks), end=Coordinate(959, ticks), source='dlt_2yklvx93ukbwa'),
 TimeIntervalStamp(start=Coordinate(960, ticks), end=Coordinate(1199, ticks), source='dlt_2yklvx93ukbwa'),
 TimeIntervalStamp(start=Coordinate(1200, ticks), end=Coordinate(1679, ticks), source='dlt_2yklvx93ukbwa'),
 TimeIntervalStamp(start=Coordinate(1680, ticks), end=Coordinate(1919, ticks), source='dlt_2yklvx93ukbwa')]

In [102]:
score_midi_tl.get_timestamp_table()

pyarrow.Table
dlt_2yklvx93ukbwa: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
ticks_to_quarters: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
ticks_to_seconds: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
dlt_2yklvx93ukbwa: [
  -- is_valid: all not null
  -- child 0 type: double
[0,239,240,959,960,...,5999,6000,6479,6480,6959]
  -- child 1 type: int64
[0,239,240,959,960,...,5999,6000,6479,6480,6959]
  -- child 2 type: int64
[1,1,1,1,1,...,1,1,1,1,1]]
ticks_to_quarters: [
  -- is_valid: all not null
  -- child 0 type: double
[0,0.4979166666666667,0.5,1.9979166666666666,2,...,12.497916666666667,12.5,13.497916666666667,13.5,14.497916666666667]
  -- child 1 type: int64
[0,4484834628923119,1,89978

***
## 7. TabularLoaders (CsvLoader, TsvLoader, LabLoader)

Generic tabular loaders with configurable column mapping.
The score `Ms3Loader` above dispatches ms3 facets by filename into a
`ScoreStore`; this section covers the generic tabular loaders.

### 7a. Ms3Loader score facets

In [103]:
from timetoalign import Ms3Loader

# A score-facet filename selects the notes facet in the resulting ScoreStore.
# Use an ms3-style annotation TSV (unfolded notes).
ms3_file = (
    SCORE_DIR
    / "flow_control"
    / "polyrythm_only"
    / "out_of_the_flow_experience-polyrhythm_only.notes.tsv"
)
ms3_file.name

'out_of_the_flow_experience-polyrhythm_only.notes.tsv'

In [104]:
ms3_loader = Ms3Loader()
ms3_loader.load(ms3_file)
ms3_loader

Sources,1 file(s): out_of_the_flow_experience-polyrhythm_only.notes.tsv
Events,241
Unit,quarters
Try,"create_timeline(), get_events(...)"


In [105]:
ms3_loader.store

ScoreStore(notes=241, measures=0, controls=0, annotations=0)

In [106]:
ms3_loader.store.notes  # NoteEventData in the ScoreStore

Events,241
Unit,quarters
Number type,fraction
Fields,midi : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [107]:
ms3_loader.store.notes.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
mc: int64
  -- field metadata --
  number_type: 'int64'
mn: string
mc_onset: struct<value: double, numerator: int64, denominator: int64>
  child 0, val

In [108]:
ms3_loader.store.notes.to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,mc,mn,mc_onset,...,tpc,octave,tied,gracenote,chord_id,voice,staff,part_id,locator,parse_id
0,note_335nnl3a39yst,A♭1,interval,Note,0,3/20,3/20,1,0,0,...,-4,1,NaN,NaN,16,1,4,P1,row:0,parse_1nmvcw0ra670f
1,note_1oofxk41gje9c,C♭3,interval,Note,0,1/4,1/4,1,0,0,...,-7,3,NaN,NaN,12,1,3,P1,row:1,parse_1nmvcw0ra670f
2,note_19hzn2kqqmhm7,E♭3,interval,Note,0,1/4,1/4,1,0,0,...,-3,3,NaN,NaN,7,1,2,P1,row:2,parse_1nmvcw0ra670f
3,note_0ms88bbu8dy1s,C♭6,interval,Note,0,3/14,3/14,1,0,0,...,-7,6,NaN,NaN,0,1,1,P1,row:3,parse_1nmvcw0ra670f
4,note_2ertyoi3lnj8m,E♭2,interval,Note,3/20,3/10,3/20,1,0,3/80,...,-3,2,NaN,NaN,17,1,4,P1,row:4,parse_1nmvcw0ra670f


In [109]:
ms3_tl = ms3_loader.create_timeline()
ms3_tl

ContinuousLogicalTimeline(id='clt_37muy2hrwkbku', length=93/2, unit=quarters, events=241, children=0)

In [110]:
ms3_tl.get_events()

Events,241
Unit,quarters
Number type,fraction
Fields,midi : EnharmonicPitch
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [111]:
ms3_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
mc: int64
  -- field metadata --
  number_type: 'int64'
mn: string
mc_onset: struct<value: double, numerator: int64, denominator: int64>
  child 0, val

In [112]:
ms3_first_event_id = first_id(ms3_tl)
ms3_tl.get_event(ms3_first_event_id)

{'id': 'note_335nnl3a39yst',
 'name': 'A♭1',
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 0.15, 'numerator': 3, 'denominator': 20},
 'duration': {'value': 0.15, 'numerator': 3, 'denominator': 20},
 'mc': 1,
 'mn': '0',
 'mc_onset': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'mn_onset': {'value': 1.25, 'numerator': 5, 'denominator': 4},
 'specific_pitch': {'step': 'A', 'alter': -1, 'octave': 1, 'cents': 0.0},
 'midi': 32,
 'tpc': -4,
 'octave': 1,
 'tied': None,
 'gracenote': None,
 'chord_id': '16',
 'voice': 1,
 'staff': 4,
 'part_id': 'P1'}

In [113]:
ms3_tl.get_timestamp_for(ms3_first_event_id)

TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(3, 20), quarters), source='clt_37muy2hrwkbku')

In [114]:
ms3_tl.get_timestamp_at(0)

ID,Coordinate,Type
clt_37muy2hrwkbku,0 quarters,axis


In [115]:
ms3_tl.get_timestamp_table(format="dataframe")

,clt_37muy2hrwkbku (quarters)
id,
note_335nnl3a39yst,0
note_2ertyoi3lnj8m,3/20
note_0f578mfocux4i,3/14
note_117hie55ceyal,1/4
note_01f3nnuhlshtw,3/10
...,...
note_0jsg5kaon86ra,635/14
note_1jxu811rzpsi4,91/2
,183/4


In [116]:
ms3_event_ids = first_ids(ms3_tl)
ms3_tl.get_timestamps_for(ms3_event_ids)

[TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(3, 20), quarters), source='clt_37muy2hrwkbku'),
 TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 4), quarters), source='clt_37muy2hrwkbku'),
 TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 4), quarters), source='clt_37muy2hrwkbku'),
 TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(3, 14), quarters), source='clt_37muy2hrwkbku'),
 TimeIntervalStamp(start=Coordinate(Fraction(3, 20), quarters), end=Coordinate(Fraction(3, 10), quarters), source='clt_37muy2hrwkbku')]

In [117]:
ms3_tl.get_timestamp_table()

pyarrow.Table
clt_37muy2hrwkbku: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
clt_37muy2hrwkbku: [
  -- is_valid: all not null
  -- child 0 type: double
[0,0.15,0.21428571428571427,0.25,0.3,...,45.357142857142854,45.5,45.75,45.92857142857143,46.5]
  -- child 1 type: int64
[0,3,3,1,3,...,635,91,183,643,93]
  -- child 2 type: int64
[1,20,14,4,10,...,14,2,4,14,2]]

### 7b. CsvLoader

In [118]:
from timetoalign.loader.tabular.csv import CsvLoader

csv_file = DATA_DIR / "tabular" / "test_events.csv"
csv_file.name

'test_events.csv'

In [119]:
csv_loader = CsvLoader()
csv_loader.load(csv_file)
csv_loader

Sources,1 file(s): test_events.csv
Events,5
Unit,seconds
Try,"create_timeline(), get_events(...)"


In [120]:
csv_loader.events.to_dataframe()

,id,name,temporal_type,event_type,start,end,duration,label,locator,parse_id
0,evt001,NaN,interval,Note,0.0,1.0,1.0,first_note,row:0 key:evt001,parse_35lr32y9kt7bo
1,evt002,NaN,interval,Note,1.0,2.0,1.0,second_note,row:1 key:evt002,parse_35lr32y9kt7bo
2,evt003,NaN,interval,Note,2.0,3.5,1.5,third_note,row:2 key:evt003,parse_35lr32y9kt7bo
3,evt004,NaN,interval,Rest,3.5,4.0,0.5,rest,row:3 key:evt004,parse_35lr32y9kt7bo
4,evt005,NaN,interval,Note,4.0,5.0,1.0,last_note,row:4 key:evt005,parse_35lr32y9kt7bo


Reaching past `to_dataframe()` to the backing Arrow table shows how a
coordinate is actually stored: each cell is a struct carrying `value` together
with the exact `numerator` and `denominator`. That is what keeps an authored
ratio exact in storage, and it is why `to_dataframe()` — which resolves each
struct to one scalar — is the right call for reading and display.

In [121]:
csv_loader.events.table.to_pandas()

,id,name,temporal_type,event_type,start,end,duration,label,locator,parse_id
0,evt001,NaN,interval,Note,"{'value': 0.0, 'numerator': 0, 'denominator': 1}","{'value': 1.0, 'numerator': 1, 'denominator': 1}","{'value': 1.0, 'numerator': 1, 'denominator': 1}",first_note,row:0 key:evt001,parse_35lr32y9kt7bo
1,evt002,NaN,interval,Note,"{'value': 1.0, 'numerator': 1, 'denominator': 1}","{'value': 2.0, 'numerator': 2, 'denominator': 1}","{'value': 1.0, 'numerator': 1, 'denominator': 1}",second_note,row:1 key:evt002,parse_35lr32y9kt7bo
2,evt003,NaN,interval,Note,"{'value': 2.0, 'numerator': 2, 'denominator': 1}","{'value': 3.5, 'numerator': 7, 'denominator': 2}","{'value': 1.5, 'numerator': 3, 'denominator': 2}",third_note,row:2 key:evt003,parse_35lr32y9kt7bo
3,evt004,NaN,interval,Rest,"{'value': 3.5, 'numerator': 7, 'denominator': 2}","{'value': 4.0, 'numerator': 4, 'denominator': 1}","{'value': 0.5, 'numerator': 1, 'denominator': 2}",rest,row:3 key:evt004,parse_35lr32y9kt7bo
4,evt005,NaN,interval,Note,"{'value': 4.0, 'numerator': 4, 'denominator': 1}","{'value': 5.0, 'numerator': 5, 'denominator': 1}","{'value': 1.0, 'numerator': 1, 'denominator': 1}",last_note,row:4 key:evt005,parse_35lr32y9kt7bo


In [122]:
csv_loader.events.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
label: large_string
locator: string
parse_id: string
-- schema metadata --
timetoalign: '{"loader_class": "EventData", "number_type": "float", "sour' + 74

In [123]:
csv_loader.events.to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,label,locator,parse_id
0,evt001,NaN,interval,Note,0.0,1.0,1.0,first_note,row:0 key:evt001,parse_35lr32y9kt7bo
1,evt002,NaN,interval,Note,1.0,2.0,1.0,second_note,row:1 key:evt002,parse_35lr32y9kt7bo
2,evt003,NaN,interval,Note,2.0,3.5,1.5,third_note,row:2 key:evt003,parse_35lr32y9kt7bo
3,evt004,NaN,interval,Rest,3.5,4.0,0.5,rest,row:3 key:evt004,parse_35lr32y9kt7bo
4,evt005,NaN,interval,Note,4.0,5.0,1.0,last_note,row:4 key:evt005,parse_35lr32y9kt7bo


In [124]:
csv_tl = csv_loader.create_timeline()
csv_tl

ContinuousPhysicalTimeline(id='cpt_2kslzdwlaxn0v', length=5.0, unit=seconds, events=5, children=0)

In [125]:
csv_tl.get_events()

Events,5
Unit,seconds
Number type,float
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [126]:
csv_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
label: large_string
-- schema metadata --
timetoalign: '{"loader_class": "EventData", "number_type": "float", "sour' + 74

In [127]:
csv_first_event_id = first_id(csv_tl)
csv_tl.get_event(csv_first_event_id)

{'id': 'evt001',
 'name': None,
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'duration': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'label': 'first_note'}

In [128]:
csv_tl.get_timestamp_for(csv_first_event_id)

TimeIntervalStamp(start=Coordinate(0.0, seconds), end=Coordinate(1.0, seconds), source='cpt_2kslzdwlaxn0v')

In [129]:
csv_tl.get_timestamp_at(0)

ID,Coordinate,Type
cpt_2kslzdwlaxn0v,0 seconds,axis


In [130]:
csv_tl.get_timestamp_table(format="dataframe")

,cpt_2kslzdwlaxn0v (seconds)
id,
evt001,0.0
evt002,1.0
evt003,2.0
evt004,3.5
evt005,4.0
,5.0


In [131]:
csv_event_ids = first_ids(csv_tl)
csv_tl.get_timestamps_for(csv_event_ids)

[TimeIntervalStamp(start=Coordinate(0.0, seconds), end=Coordinate(1.0, seconds), source='cpt_2kslzdwlaxn0v'),
 TimeIntervalStamp(start=Coordinate(1.0, seconds), end=Coordinate(2.0, seconds), source='cpt_2kslzdwlaxn0v'),
 TimeIntervalStamp(start=Coordinate(2.0, seconds), end=Coordinate(3.5, seconds), source='cpt_2kslzdwlaxn0v'),
 TimeIntervalStamp(start=Coordinate(3.5, seconds), end=Coordinate(4.0, seconds), source='cpt_2kslzdwlaxn0v'),
 TimeIntervalStamp(start=Coordinate(4.0, seconds), end=Coordinate(5.0, seconds), source='cpt_2kslzdwlaxn0v')]

In [132]:
csv_tl.get_timestamp_table()

pyarrow.Table
cpt_2kslzdwlaxn0v: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
cpt_2kslzdwlaxn0v: [
  -- is_valid: all not null
  -- child 0 type: double
[0,1,2,3.5,4,5]
  -- child 1 type: int64
[0,1,2,7,4,5]
  -- child 2 type: int64
[1,1,1,2,1,1]]

### 7c. LabLoader

Parses headerless tab-separated files in Audacity/Praat label format.
**No `.lab` test data currently exists in the test suite -- this is a gap.**

In [133]:
from timetoalign.loader.physical.audio import AudioLoader

In [134]:
# LabLoader example (test data in tests/data/fixtures/lab/)
# from timetoalign.loader.tabular.csv import LabLoader
# lab_loader = LabLoader()
# lab_loader.load("regions.lab")
# lab_loader

***
## 8. AudioLoader

Reads audio file metadata (sample rate, channels, duration) without loading
the full waveform.  Produces a `DiscretePhysicalTimeline` in samples with a
`SamplesToSeconds` C-Map.

In [135]:
audio_file = DATA_DIR / "supra" / "midi" / "fd660zf8362.mp3"
audio_file.name

'fd660zf8362.mp3'

In [136]:
audio_loader = AudioLoader()
audio_loader.load(audio_file)
audio_loader

AudioLoader(samples=19715328, rate=44100Hz, duration=447.06s, format=MP3)

In [137]:
audio_loader.audio_info

AudioInfo(n_samples=19715328, sample_rate=44100, channels=2, duration_seconds=447.0595918367347, format='MP3', subtype='MPEG_LAYER_III', bits_per_sample=None, source_path=PosixPath('/home/laser/git/tta/timetoalign/tests/data/supra/midi/fd660zf8362.mp3'), extra={'sections': 1}, backend='soundfile')

In [138]:
print(f"Sample rate: {audio_loader.sample_rate}")
print(f"Channels: {audio_loader.channels}")
print(f"Duration: {audio_loader.duration_seconds:.2f}s")
print(f"Samples: {audio_loader.n_samples}")

Sample rate: 44100
Channels: 2
Duration: 447.06s
Samples: 19715328


In [139]:
audio_tl = audio_loader.create_timeline()
audio_tl

DiscretePhysicalTimeline(id='dpt_3qo5n1ufmdwfu', length=19715328, unit=samples, events=0, children=0, cmaps=1)

`create_continuous_timeline()` gives the same file as a
`ContinuousPhysicalTimeline` in seconds with a `SecondsToSamples` C-Map.

In [140]:
audio_loader.create_continuous_timeline()

ContinuousPhysicalTimeline(id='cpt_1oel4ws65dm7m', length=447.0595918367347, unit=seconds, events=0, children=0, cmaps=1)

***
## 9. EepNotesLoader (EEP .notes alignment files)

Subclass of `CsvLoader` for whitespace-separated EEP `.notes` files
containing note-level performance alignments.

In [141]:
from timetoalign.loader.physical.eep_notes import EepNotesLoader

eep_file = (
    SCORE_DIR
    / "beethoven_op18-4iv_multimodal"
    / "StringQuartetEEP_I_Exaggerated"
    / "StringQuartetEEP_I_Exaggerated_align_cello.notes"
)
eep_file.name

'StringQuartetEEP_I_Exaggerated_align_cello.notes'

In [142]:
eep_loader = EepNotesLoader()
eep_loader.load(eep_file)
eep_loader

Sources,1 file(s): StringQuartetEEP_I_Exaggerated_align_cello.notes
Events,591
Unit,seconds
Try,"create_timeline(), get_events(...)"


In [143]:
eep_loader.store

Table,Unit,Range,Events
events,seconds,1.00 – 184.69,591


In [144]:
eep_loader.events

Events,591
Unit,seconds
Number type,float
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [145]:
eep_loader.events.schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
pitch: string
  -- field metadata --
  timetoalign: '{"field_type": "StringField", "version": 1}'
staff: int64
  -- field metadata --
  timetoalign: '{"field_type"

In [146]:
eep_loader.events.to_dataframe().head()

,id,name,temporal_type,event_type,start,end,duration,pitch,staff,locator,parse_id
0,note_33lr15k0h3e0d,rest,interval,Note,1.000000,1.289796,0.289796,rest,4,row:0,parse_1kp0hpf3sks9w
1,note_3cqlurkmcb80u,C3,interval,Note,1.289796,1.505805,0.216009,C3,4,row:1,parse_1kp0hpf3sks9w
2,note_38d3s2t79914v,rest,interval,Note,1.505805,1.693243,0.187438,rest,4,row:2,parse_1kp0hpf3sks9w
3,note_3qy7gqqeh5zug,C3,interval,Note,1.693243,1.909252,0.216009,C3,4,row:3,parse_1kp0hpf3sks9w
4,note_3oz48z56cr9yn,rest,interval,Note,1.909252,2.134422,0.225170,rest,4,row:4,parse_1kp0hpf3sks9w


In [147]:
eep_tl = eep_loader.create_timeline()
eep_tl

ContinuousPhysicalTimeline(id='cpt_11fe5t151aodo', length=184.787665, unit=seconds, events=591, children=0)

In [148]:
eep_tl.get_events()

Events,591
Unit,seconds
Number type,float
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [149]:
eep_tl.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
pitch: string
  -- field metadata --
  timetoalign: '{"field_type": "StringField", "version": 1}'
staff: int64
  -- field metadata --
  timetoalign: '{"field_type"

In [150]:
eep_tl.get_events(temporal_type="interval")

Events,591
Unit,seconds
Number type,float
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [151]:
eep_first_event_id = first_id(eep_tl)
eep_tl.get_event(eep_first_event_id)

{'id': 'note_33lr15k0h3e0d',
 'name': 'rest',
 'temporal_type': 'interval',
 'event_type': 'Note',
 'start': {'value': 1.0, 'numerator': 1, 'denominator': 1},
 'end': {'value': 1.289796,
  'numerator': 1452181196245989,
  'denominator': 1125899906842624},
 'duration': {'value': 0.28979599999999994,
  'numerator': 326281289403365,
  'denominator': 1125899906842624},
 'pitch': 'rest',
 'staff': 4}

In [152]:
eep_tl.get_timestamp_for(eep_first_event_id)

TimeIntervalStamp(start=Coordinate(1.0, seconds), end=Coordinate(1.289796, seconds), source='cpt_11fe5t151aodo')

In [153]:
eep_tl.get_timestamp_at(1.0)

ID,Coordinate,Type
cpt_11fe5t151aodo,1 seconds,axis


In [154]:
eep_tl.get_events_at(1.0)

{'cpt_11fe5t151aodo': [{'id': 'note_33lr15k0h3e0d',
   'name': 'rest',
   'temporal_type': 'interval',
   'event_type': 'Note',
   'start': Fraction(1, 1),
   'end': Fraction(1452181196245989, 1125899906842624),
   'duration': Fraction(326281289403365, 1125899906842624),
   'pitch': 'rest',
   'staff': 4}]}

In [155]:
eep_tl.get_timestamp_table(format="dataframe")

,cpt_11fe5t151aodo (seconds)
id,
note_33lr15k0h3e0d,1.000000
note_3cqlurkmcb80u,1.289796
note_38d3s2t79914v,1.505805
note_3qy7gqqeh5zug,1.693243
note_3oz48z56cr9yn,1.909252
...,...
note_2vx3o0mgeuny9,184.033742
note_22kq6677xvb7v,184.246486
note_3nu6uypa929dz,184.474921


In [156]:
eep_event_ids = first_ids(eep_tl)
eep_tl.get_timestamps_for(eep_event_ids)

[TimeIntervalStamp(start=Coordinate(1.0, seconds), end=Coordinate(1.289796, seconds), source='cpt_11fe5t151aodo'),
 TimeIntervalStamp(start=Coordinate(1.289796, seconds), end=Coordinate(1.505805, seconds), source='cpt_11fe5t151aodo'),
 TimeIntervalStamp(start=Coordinate(1.505805, seconds), end=Coordinate(1.693243, seconds), source='cpt_11fe5t151aodo'),
 TimeIntervalStamp(start=Coordinate(1.693243, seconds), end=Coordinate(1.909252, seconds), source='cpt_11fe5t151aodo'),
 TimeIntervalStamp(start=Coordinate(1.909252, seconds), end=Coordinate(2.134422, seconds), source='cpt_11fe5t151aodo')]

In [157]:
eep_tl.get_timestamp_table()

pyarrow.Table
cpt_11fe5t151aodo: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
cpt_11fe5t151aodo: [
  -- is_valid: all not null
  -- child 0 type: double
[1,1.289796,1.505805,1.693243,1.909252,...,184.033742,184.246486,184.474921,184.687665,184.787665]
  -- child 1 type: int64
[1,1452181196245989,3390771418446315,7625688543847701,4299253297878187,...,6475111655428109,50645288433467,202832320672559,6498119525577555,3250818981393219]
  -- child 2 type: int64
[1,1125899906842624,2251799813685248,4503599627370496,2251799813685248,...,35184372088832,274877906944,1099511627776,35184372088832,17592186044416]]

***
## 10. RepoVizzLoader (RepoVizz sensor CSV)

A `ManifestLoader` for 2-line CSV files from the RepoVizz platform
(MoCap/sensor data).  Returns metadata (frame rate, samples), not events.

**Note:** `RepoVizzLoader` is a `ManifestLoader`, not a `Loader`.
It has no `.store` or `.events` -- only `.create_timeline()`.

In [158]:
from timetoalign.loader.physical.repovizz import RepoVizzLoader

repovizz_dir = (
    SCORE_DIR / "beethoven_op18-4iv_multimodal" / "StringQuartetEEP_I_Mechanical"
)
repovizz_files = sorted(repovizz_dir.glob("*.csv"))[:1]  # just one sensor file
repovizz_files[0].name if repovizz_files else "NO CSV FILES FOUND"

'celloBowFrogL_Xcoord.csv'

In [159]:
if repovizz_files:
    repovizz_loader = RepoVizzLoader()
    repovizz_loader.load(repovizz_files[0])
    repovizz_loader

In [160]:
if repovizz_files:
    print(f"Frame rate: {repovizz_loader.frame_rate}")
    print(f"Samples: {repovizz_loader.n_samples}")
    print(f"Duration: {repovizz_loader.duration_seconds:.2f}s")

Frame rate: 240
Samples: 67628
Duration: 281.78s


In [161]:
if repovizz_files:
    repovizz_tl = repovizz_loader.create_timeline()
    repovizz_tl

***
## 11. GraphicalLoader (images / PDF pages)

Factory class for building graphical timelines from image sources and
path segments. Uses a builder pattern: `.add_image()`, `.add_horizontal_segment()`,
then `.store` to produce a `GraphicalStore`.

In [162]:
from timetoalign.loader.graphical.loader import GraphicalLoader

thoresen_dir = DATA_DIR / "thoresen"
thoresen_images = sorted(thoresen_dir.glob("*.jpeg"))[:2]
[img.name for img in thoresen_images] if thoresen_images else "NO IMAGES FOUND"

['thoresen_2009_sound-objects_p312_page1_1.jpeg',
 'thoresen_2010_form-building-patterns_p90-91_page1_1.jpeg']

In [163]:
if thoresen_images:
    graphical_loader = GraphicalLoader()
    src_idx = graphical_loader.add_image(thoresen_images[0])
    graphical_loader.add_horizontal_segment(src_idx, x0=0, x1=500, y=100)
    graphical_loader

In [164]:
if thoresen_images:
    graphical_store = graphical_loader.store
    graphical_store

***
## 12. IIIFManifestLoader (IIIF JSON manifests)

Parses IIIF Presentation API manifests to extract image dimensions
and metadata for graphical timeline construction.

**Note:** Standalone class, not a `Loader` subclass.

In [165]:
from timetoalign.loader.graphical.iiif import IIIFManifestLoader

iiif_file = DATA_DIR / "supra" / "image" / "ifff_manifest.json"
iiif_file.name

'ifff_manifest.json'

In [166]:
iiif_loader = IIIFManifestLoader()
iiif_loader.load(iiif_file)
iiif_loader

IIIFManifestLoader(canvases=1, dimensions=4096x299400)

In [167]:
print(f"Label: {iiif_loader.label}")
print(f"Canvases: {iiif_loader.n_canvases}")
print(f"Dimensions: {iiif_loader.dimensions}")

Label: Meistersinger von Nürnberg : Vorspiel
Canvases: 1
Dimensions: {'width': 4096, 'height': 299400}


An image has no time axis until its path through the image is given.
The manifest describes the scan of a piano roll, which is read from top
to bottom: its time path is a vertical line down the middle of the
image, the whole width as its strip. `create_timelines()` answers one
timeline per canvas along the path given; here there is one canvas.

In [168]:
from timetoalign.loader.graphical import VerticalLinePath

roll_path = VerticalLinePath(x=2048, y0=0, y1=299400, tolerance=2048)
(iiif_tl,) = iiif_loader.create_timelines(roll_path)
iiif_tl

DiscreteGraphicalTimeline(id='dgt_0i6cl4atj75s6', length=299400, unit=pixels, events=0, children=0)

***
## 13. ATONLoader (piano roll analysis)

Parses ATON format files describing piano roll hole positions.

In [169]:
from timetoalign.loader.format.json import JsonLoader

In [170]:
# ATONLoader example (test data in tests/data/fixtures/aton/)
# from timetoalign.loader.graphical.aton import ATONLoader
# aton_loader = ATONLoader()
# aton_loader.load("minimal.aton")
# aton_loader

***
## 14. JsonLoader (generic JSON normaliser)

Configurable JSON loader that flattens nested structures into PyArrow tables.
Base class for format-specific loaders like `TiliaJsonLoader`.

In [171]:
# Use the TiLiA JSON as a generic JSON example
json_file = SCORE_DIR / "bruckner5_scherzo" / "harnoncourt" / "Bruckner5_Scherzo.json"
json_file.name if json_file.exists() else "NOT FOUND"

'Bruckner5_Scherzo.json'

In [172]:
if json_file.exists():
    json_loader = JsonLoader()
    json_loader.load(json_file)
    json_loader

In [173]:
if json_file.exists():
    json_loader.store

In [174]:
if json_file.exists():
    json_loader.keys()

In [175]:
# JsonLoader has .get_table() -- good, this should be on all loaders
if json_file.exists() and json_loader.keys():
    first_key = json_loader.keys()[0]
    json_loader.get_table(first_key)

***
## 15. TiliaJsonLoader (TiLiA .tla/.json analysis)

Subclass of `JsonLoader` specialised for TiLiA timeline analysis exports.
Produces timelines, groups, and alignment bundles.

**This is the most feature-complete loader in terms of the target API.**

In [176]:
from timetoalign.loader.alignment.tilia import TiliaJsonLoader

tilia_file = SCORE_DIR / "bruckner5_scherzo" / "harnoncourt" / "Bruckner5_Scherzo.json"
tilia_file.name

'Bruckner5_Scherzo.json'

In [177]:
tilia_loader = TiliaJsonLoader()
tilia_loader.load(tilia_file)
tilia_loader

Sources,1 file(s): Bruckner5_Scherzo.json
Events,7
Unit,seconds
Try,"create_timeline(), create_group(), create_bundle(), get_events(...)"


In [178]:
tilia_loader.store

Table,Unit,Range,Events
HIERARCHY_TIMELINE_0,seconds,0.00 – 0.00,33
MARKER_TIMELINE_1,seconds,0.00 – 0.00,14
MARKER_TIMELINE_2,seconds,0.00 – 0.00,5
BEAT_TIMELINE_3,seconds,0.00 – 0.00,1146
MARKER_TIMELINE_4,seconds,0.00 – 0.00,12
MARKER_TIMELINE_5,seconds,0.00 – 0.00,11
PDF_TIMELINE_6,seconds,0.00 – 0.00,19


In [179]:
tilia_loader.store.keys()  # TiliaDictStore

('HIERARCHY_TIMELINE_0',
 'MARKER_TIMELINE_1',
 'MARKER_TIMELINE_2',
 'BEAT_TIMELINE_3',
 'MARKER_TIMELINE_4',
 'MARKER_TIMELINE_5',
 'PDF_TIMELINE_6')

In [180]:
tilia_loader.timeline_ids

['cpt_2hygiikvjca99',
 'cpt_0rzdkw0b3u89g',
 'cpt_039470m6nvrsj',
 'cpt_3ad3dapt9xwt8',
 'cpt_2mzr720vpk0v0',
 'cpt_14hwvq4kx7jat',
 'cpt_12kb3xeaieoj6']

TiLiA timelines receive minted IDs (`cpt_…`), seeded from the file the loader
read. The source timeline name remains available for lookup — for example,
`BEAT_TIMELINE_3` can still select that source timeline — but it is not the
stored ID.

In [181]:
tilia_loader.timeline_specs

[{'id': 'cpt_2hygiikvjca99',
  'source_id': 'HIERARCHY_TIMELINE_0',
  'kind': 'HIERARCHY_TIMELINE',
  'name': 'Form (Harnoncourt)',
  'n_components': 33,
  'ordinal': 2,
  'index': 0,
  'raw': {'kind': 'HIERARCHY_TIMELINE',
   'name': 'Form (Harnoncourt)',
   'height': 160,
   'is_visible': True,
   'ordinal': 2,
   'components': [{'start': 48.401313588026454,
     'pre_start': 48.401313588026454,
     'end': 72.61305236816406,
     'post_end': 72.61305236816406,
     'level': 1,
     'label': 'Forts. der II. Th. gruppe',
     'formal_type': '',
     'formal_function': '',
     'comments': '"auch im schnellen Tempo teil von 2. Th"',
     'start_measure': 47,
     'start_beat': 1,
     'end_measure': 79,
     'end_beat': 3,
     'length': 24.21173878013761,
     'length_in_measures': [32, 2],
     'kind': 'HIERARCHY'},
    {'start': 86.1124496459961,
     'pre_start': 86.1124496459961,
     'end': 96.7035903930664,
     'post_end': 96.7035903930664,
     'level': 1,
     'label': 'OP. E

In [182]:
# TiliaJsonLoader has create_timeline, create_timelines, create_group -- good!
tilia_tls = tilia_loader.create_timelines()
{i: tl for i, tl in enumerate(tilia_tls)}

{0: ContinuousPhysicalTimeline(id='cpt_2hygiikvjca99', length=788.0, unit=seconds, events=33, children=0),
 1: ContinuousPhysicalTimeline(id='cpt_0rzdkw0b3u89g', length=788.0, unit=seconds, events=14, children=0),
 2: ContinuousPhysicalTimeline(id='cpt_039470m6nvrsj', length=788.0, unit=seconds, events=5, children=0),
 3: ContinuousPhysicalTimeline(id='cpt_3ad3dapt9xwt8', length=788.0, unit=seconds, events=1146, children=0),
 4: ContinuousPhysicalTimeline(id='cpt_2mzr720vpk0v0', length=788.0, unit=seconds, events=12, children=0),
 5: ContinuousPhysicalTimeline(id='cpt_14hwvq4kx7jat', length=788.0, unit=seconds, events=11, children=0),
 6: ContinuousPhysicalTimeline(id='cpt_12kb3xeaieoj6', length=788.0, unit=seconds, events=19, children=0)}

In [183]:
tilia_tl0 = tilia_loader.create_timeline(uid=tilia_loader.timeline_ids[0])
tilia_tl0

ContinuousPhysicalTimeline(id='cpt_2hygiikvjca99', length=788.0, unit=seconds, events=33, children=0)

In [184]:
tilia_tl0.get_events()

Events,33
Unit,seconds
Number type,float
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [185]:
tilia_tl0.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "float", "unit": "seconds", "version": 1}'
comments: string
end_beat: int64
end_measure: int64
formal_function: string
formal_type: string
kind: string
length: double
length_in_measures: list<item: int64>
 

In [186]:
tilia_first_event_id = first_id(tilia_tl0)
tilia_tl0.get_event(tilia_first_event_id)

{'id': 'hierarchy_261guhxuszzbr',
 'name': 'Forts. der II. Th. gruppe',
 'temporal_type': 'interval',
 'event_type': 'Hierarchy',
 'start': {'value': 48.401313588026454,
  'numerator': 1702969826869363,
  'denominator': 35184372088832},
 'end': {'value': 72.61305236816406,
  'numerator': 4758769,
  'denominator': 65536},
 'duration': {'value': 24.21173878013761,
  'numerator': 851874826157965,
  'denominator': 35184372088832},
 'comments': '"auch im schnellen Tempo teil von 2. Th"',
 'end_beat': 3,
 'end_measure': 79,
 'formal_function': '',
 'formal_type': '',
 'kind': 'HIERARCHY',
 'length': 24.21173878013761,
 'length_in_measures': [32, 2],
 'level': 1,
 'post_end': 72.61305236816406,
 'pre_start': 48.401313588026454,
 'start_beat': 1,
 'start_measure': 47}

In [187]:
tilia_tl0.get_timestamp_for(tilia_first_event_id)

TimeIntervalStamp(start=Coordinate(48.401313588026454, seconds), end=Coordinate(72.61305236816406, seconds), source='cpt_2hygiikvjca99')

In [188]:
tilia_tl0.get_timestamp_at(10)

ID,Coordinate,Type
cpt_2hygiikvjca99,10 seconds,axis


In [189]:
tilia_tl0.get_events_at(10)

{'cpt_2hygiikvjca99': [{'id': 'hierarchy_2ldey6t1hukqc',
   'name': 'I. Th.',
   'temporal_type': 'interval',
   'event_type': 'Hierarchy',
   'start': Fraction(4365754772616025, 9007199254740992),
   'end': Fraction(8848788725706453, 562949953421312),
   'duration': Fraction(8575929052417951, 562949953421312),
   'comments': '',
   'end_beat': 1,
   'end_measure': 23,
   'formal_function': '',
   'formal_type': '',
   'kind': 'HIERARCHY',
   'length': 15.233910226473936,
   'length_in_measures': [22, 0],
   'level': 2,
   'post_end': 15.718606373316485,
   'pre_start': 0.4846961468425476,
   'start_beat': 1,
   'start_measure': 1},
  {'id': 'hierarchy_2gd0ngka6ihxc',
   'name': 'Expos',
   'temporal_type': 'interval',
   'event_type': 'Hierarchy',
   'start': Fraction(4365754772616025, 9007199254740992),
   'end': Fraction(7916298577329755, 70368744177664),
   'duration': Fraction(1970547779542173, 17592186044416),
   'comments': '',
   'end_beat': 3,
   'end_measure': 132,
   'formal

In [190]:
tilia_event_ids = first_ids(tilia_tl0)
tilia_tl0.get_timestamps_for(tilia_event_ids)

[TimeIntervalStamp(start=Coordinate(48.401313588026454, seconds), end=Coordinate(72.61305236816406, seconds), source='cpt_2hygiikvjca99'),
 TimeIntervalStamp(start=Coordinate(86.1124496459961, seconds), end=Coordinate(96.7035903930664, seconds), source='cpt_2hygiikvjca99'),
 TimeIntervalStamp(start=Coordinate(96.7035903930664, seconds), end=Coordinate(112.49736896459346, seconds), source='cpt_2hygiikvjca99'),
 TimeIntervalStamp(start=Coordinate(127.53980255126953, seconds), end=Coordinate(136.32819401716614, seconds), source='cpt_2hygiikvjca99'),
 TimeIntervalStamp(start=Coordinate(140.86165148789726, seconds), end=Coordinate(144.7765655517578, seconds), source='cpt_2hygiikvjca99')]

In [191]:
tilia_tl0.get_timestamp_table()

pyarrow.Table
cpt_2hygiikvjca99: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
cpt_2hygiikvjca99: [
  -- is_valid: all not null
  -- child 0 type: double
[0.4846961468425476,15.718606373316485,48.401313588026454,72.61305236816406,86.1124496459961,...,407.3296758653347,421.0113220214844,435.818115234375,455.1636272157131,787]
  -- child 1 type: int64
[4365754772616025,8848788725706453,1702969826869363,4758769,11286931,...,3582909719617317,13795699,1785111,8007323210630035,787]
  -- child 2 type: int64
[9007199254740992,562949953421312,35184372088832,65536,131072,...,8796093022208,32768,4096,17592186044416,1]]

### TiliaJsonLoader: TimelineGroup

In [192]:
tilia_group = tilia_loader.create_group()
tilia_group

TimelineGroup(id='group_1nzkheefpprq1', n_timelines=7)

In [193]:
len(tilia_group)  # number of timelines

7

In [194]:
tilia_group[tilia_group.timeline_ids[0]]

ContinuousPhysicalTimeline(id='cpt_2hygiikvjca99', length=788.0, unit=seconds, events=33, children=0)

In [195]:
tilia_group.get_timeline(tilia_group.timeline_ids[0])

ContinuousPhysicalTimeline(id='cpt_2hygiikvjca99', length=788.0, unit=seconds, events=33, children=0)

In [196]:
tilia_group.get_timestamp_at(10, tilia_group.timeline_ids[0])

ID,Coordinate,Type
cpt_2hygiikvjca99,10 seconds,axis
cpt_0rzdkw0b3u89g,10 seconds,child
cpt_039470m6nvrsj,10 seconds,child
cpt_3ad3dapt9xwt8,10 seconds,child
cpt_2mzr720vpk0v0,10 seconds,child
cpt_14hwvq4kx7jat,10 seconds,child
cpt_12kb3xeaieoj6,10 seconds,child


In [197]:
tilia_group.get_events().to_dataframe()

,id,name,temporal_type,event_type,start,end,duration,comments,end_beat,end_measure,...,length_in_measures,level,post_end,pre_start,start_beat,start_measure,timeline_id,beat,measure,page_number
0,hierarchy_261guhxuszzbr,Forts. der II. Th. gruppe,interval,Hierarchy,48.401314,72.613052,24.211739,"""auch im schnellen Tempo teil von 2. Th""",3.0,79.0,...,"[32, 2]",1.0,72.613052,48.401314,1.0,47.0,cpt_2hygiikvjca99,NaN,NaN,NaN
1,hierarchy_3a3jvf21abi5m,OP. E,interval,Hierarchy,86.112450,96.703590,10.591141,,2.0,112.0,...,"[15, 1]",1.0,96.703590,86.112450,1.0,97.0,cpt_2hygiikvjca99,NaN,NaN,NaN
2,hierarchy_3swgaou7o0lim,OP. A,interval,Hierarchy,96.703590,112.497369,15.793779,,3.0,132.0,...,"[20, 1]",1.0,112.497369,96.703590,2.0,112.0,cpt_2hygiikvjca99,NaN,NaN,NaN
3,hierarchy_1xc0zj45jisgq,"""1. Höhepunkt der Df""?",interval,Hierarchy,127.539803,136.328194,8.788391,,3.0,170.0,...,"[13, 0]",1.0,136.328194,127.539803,3.0,157.0,cpt_2hygiikvjca99,NaN,NaN,NaN
4,hierarchy_0wbixytaa6fca,"""Höhepunkt der Df""?",interval,Hierarchy,140.861651,144.776566,3.914914,,1.0,183.0,...,"[5, 2]",1.0,144.776566,140.861651,2.0,177.0,cpt_2hygiikvjca99,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1235,pdfmarker_2degazpr3dyk7,NaN,instant,PdfMarker,152.354198,NaN,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,NaN,cpt_12kb3xeaieoj6,2.0,188.0,15.0
1236,pdfmarker_2mjmpqh7c4pkb,NaN,instant,PdfMarker,184.481607,NaN,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,NaN,cpt_12kb3xeaieoj6,1.0,211.0,16.0
1237,pdfmarker_2om3rdky45nw2,NaN,instant,PdfMarker,210.836122,NaN,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,NaN,cpt_12kb3xeaieoj6,1.0,231.0,17.0
1238,pdfmarker_1hqlysnekscbf,NaN,instant,PdfMarker,343.025125,NaN,NaN,NaN,NaN,NaN,...,None,NaN,NaN,NaN,NaN,NaN,cpt_12kb3xeaieoj6,2.0,384.0,18.0


In [198]:
tilia_group.get_timestamp_for(tilia_first_event_id)

ID,Coordinate,Type
cpt_2hygiikvjca99,48.401313588026454 seconds,axis
cpt_0rzdkw0b3u89g,48.401313588026454 seconds,child
cpt_039470m6nvrsj,48.401313588026454 seconds,child
cpt_3ad3dapt9xwt8,48.401313588026454 seconds,child
cpt_2mzr720vpk0v0,48.401313588026454 seconds,child
cpt_14hwvq4kx7jat,48.401313588026454 seconds,child
cpt_12kb3xeaieoj6,48.401313588026454 seconds,child


In [199]:
tilia_group.get_timestamps_for(tilia_event_ids)

[TimeStamp(axis=48.401313588026454 seconds @cpt_2hygiikvjca99, source='cpt_2hygiikvjca99' (interpolated)),
 TimeStamp(axis=86.1124496459961 seconds @cpt_2hygiikvjca99, source='cpt_2hygiikvjca99' (interpolated)),
 TimeStamp(axis=96.7035903930664 seconds @cpt_2hygiikvjca99, source='cpt_2hygiikvjca99' (interpolated)),
 TimeStamp(axis=127.53980255126953 seconds @cpt_2hygiikvjca99, source='cpt_2hygiikvjca99' (interpolated)),
 TimeStamp(axis=140.86165148789726 seconds @cpt_2hygiikvjca99, source='cpt_2hygiikvjca99' (interpolated))]

In [200]:
tilia_group.get_timestamp_table()

pyarrow.Table
cpt_2hygiikvjca99: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_0rzdkw0b3u89g: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_039470m6nvrsj: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_3ad3dapt9xwt8: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_2mzr720vpk0v0: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_14hwvq4kx7jat: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_12kb3xeaie

### TiliaJsonLoader: AlignmentBundle with MatchClaim Creation

In [201]:
tilia_bundle = tilia_loader.create_bundle()
tilia_bundle

AlignmentBundle(id='bundle_0744d3mrkgsp0', name='Bruckner5_Scherzo', timelines=7, groups=1)

In [202]:
tilia_existing_claims = tilia_bundle.get_match_claims()
len(tilia_existing_claims)

0

In [203]:
if len(tilia_bundle.timeline_ids) >= 2:
    tilia_tl_a, tilia_tl_b = tilia_bundle.get_timelines(tilia_bundle.timeline_ids[:2])
    print(f"Timeline A: {tilia_tl_a.id}, Timeline B: {tilia_tl_b.id}")

Timeline A: cpt_2hygiikvjca99, Timeline B: cpt_0rzdkw0b3u89g


In [204]:
# Create MatchClaims by pairing events from two timelines
if len(tilia_bundle.timeline_ids) >= 2:
    tilia_timeline_id_a, tilia_timeline_id_b = tilia_bundle.timeline_ids[:2]
    tilia_evts_a = list(tilia_tl_a.get_events())[:2]
    tilia_evts_b = list(tilia_tl_b.get_events())[:2]
    tilia_pairs = [
        (
            tilia_evts_a[0],
            tilia_timeline_id_a,
            tilia_evts_b[0],
            tilia_timeline_id_b,
        ),
    ]
    tilia_new_claims = tilia_bundle.create_match_claims(
        tilia_pairs, synchronous=True, provenance=EXAMPLE_PROVENANCE
    )
    print(f"Created {len(tilia_new_claims)} new MatchClaim(s)")
    tilia_new_claims[0] if tilia_new_claims else "No claims created"

Created 1 new MatchClaim(s)


In [205]:
# get_matchstamp() on a MatchClaim
if len(tilia_bundle.timeline_ids) >= 2 and tilia_new_claims:
    tilia_new_claims[0].get_matchstamp()

Batch retrieval comes in two flavours. **Coordinate-batch** — "resolve THESE
coordinates" — takes a handful of query coordinates on one timeline and returns
one full cross-section per coordinate. The **whole-alignment** table further
down instead tabulates the entire alignment. Users hold coordinates, not
`MatchClaim` objects, so the coordinate-batch form is the recommended entry
point.

In [206]:
# get_matchstamps_at(coords, timeline_id) - resolve THESE coordinates into full
# MatchStamps, one per query coordinate (input order preserved). The query
# coordinates are a few of the timeline's own event coordinates.
tilia_query_tl_id = tilia_loader.timeline_ids[0]
tilia_query_events = list(tilia_tl0.get_events())
tilia_query_coords = [
    tilia_query_events[i]["start"]["value"]
    for i in (0, len(tilia_query_events) // 2, -1)
]
tilia_bundle.get_matchstamps_at(tilia_query_coords, tilia_query_tl_id)

[MatchStamp(cpt_2hygiikvjca99=48.401313588026454 seconds, cpt_0rzdkw0b3u89g=0.5089309541846749 seconds, cpt_039470m6nvrsj=0.5089309541846749 seconds, cpt_3ad3dapt9xwt8=0.5089309541846749 seconds, cpt_2mzr720vpk0v0=0.5089309541846749 seconds, cpt_14hwvq4kx7jat=0.5089309541846749 seconds, cpt_12kb3xeaieoj6=0.5089309541846749 seconds),
 MatchStamp(cpt_2hygiikvjca99=359.2872314453125 seconds, cpt_0rzdkw0b3u89g=359.2872314453125 seconds, cpt_039470m6nvrsj=359.2872314453125 seconds, cpt_3ad3dapt9xwt8=359.2872314453125 seconds, cpt_2mzr720vpk0v0=359.2872314453125 seconds, cpt_14hwvq4kx7jat=359.2872314453125 seconds, cpt_12kb3xeaieoj6=359.2872314453125 seconds),
 MatchStamp(cpt_2hygiikvjca99=0.4846961468425476 seconds, cpt_0rzdkw0b3u89g=0.4846961468425476 seconds, cpt_039470m6nvrsj=0.4846961468425476 seconds, cpt_3ad3dapt9xwt8=0.4846961468425476 seconds, cpt_2mzr720vpk0v0=0.4846961468425476 seconds, cpt_14hwvq4kx7jat=0.4846961468425476 seconds, cpt_12kb3xeaieoj6=0.4846961468425476 seconds)]

In [207]:
# get_matchstamp_table(coords, timeline_id) - the same batch as a table: one row
# per query coordinate, every connected timeline filled.
tilia_bundle.get_matchstamp_table(tilia_query_coords, tilia_query_tl_id)

pyarrow.Table
cpt_2hygiikvjca99: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_0rzdkw0b3u89g: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_039470m6nvrsj: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_3ad3dapt9xwt8: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_2mzr720vpk0v0: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_14hwvq4kx7jat: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_12kb3xeaie

In [208]:
# get_matchstamp_table() - no arguments tabulates the WHOLE alignment (one row
# per claim), a distinct operation from the coordinate-batch above.
tilia_bundle.get_matchstamp_table()

pyarrow.Table
cpt_2hygiikvjca99: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
cpt_0rzdkw0b3u89g: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
cpt_2hygiikvjca99: [
  -- is_valid: all not null
  -- child 0 type: double
[48.401313588026454]
  -- child 1 type: int64
[1702969826869363]
  -- child 2 type: int64
[35184372088832]]
cpt_0rzdkw0b3u89g: [
  -- is_valid: all not null
  -- child 0 type: double
[0.5089309541846749]
  -- child 1 type: int64
[2292021255623413]
  -- child 2 type: int64
[4503599627370496]]

***
## 16. MatchfileLoader (Vienna .match files)

Parses Vienna 4x22 corpus `.match` files via `partitura`.
Builds a shared score timeline, per-performance timelines, and `MatchClaim`s.

In [209]:
from timetoalign.loader.alignment.matchfile import MatchfileLoader

match_file = VIENNA_DIR / "Chopin_op10_no3_p22.match"
match_file.name

'Chopin_op10_no3_p22.match'

In [210]:
match_loader = MatchfileLoader()
match_loader.load(match_file)
match_loader

Sources,1 file(s): Chopin_op10_no3_p22.match
Events,1
Try,"create_timeline(), create_timelines(), create_bundle()"


In [211]:
match_tls = match_loader.create_timelines()
{tl.id: (tl.n_children, tl.n_events) for tl in match_tls}

{'score:clt_1ev0htiq1dt3p': (0, 454), 'perf:dlt_25vknh9jrada8': (0, 452)}

In [212]:
match_tl_score = match_loader.create_timeline(uid="score")
match_tl_score

ContinuousLogicalTimeline(id='score:clt_1ev0htiq1dt3p', length=83/2, unit=quarters, events=454, children=0, cmaps=2)

In [213]:
match_tl_score.get_events()

Events,454
Unit,quarters
Number type,fraction
Fields,(none)
Try,"get_field(<Scalar>), get_pitch_field(), get_raw('<col>')"


In [214]:
match_tl_score.get_events().schema

id: string not null
name: string
temporal_type: string not null
event_type: string not null
start: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
end: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
duration: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
  -- field metadata --
  timetoalign: '{"number_type": "fraction", "unit": "quarters", "version"' + 4
-- schema metadata --
timetoalign: '{"loader_class": "EventData", "number_type": "fraction", "s' + 78

In [215]:
match_first_event_id = first_id(match_tl_score)
match_tl_score.get_event(match_first_event_id)

{'id': 'n1',
 'name': None,
 'temporal_type': 'interval',
 'event_type': None,
 'start': {'value': 0.0, 'numerator': 0, 'denominator': 1},
 'end': {'value': 0.5, 'numerator': 1, 'denominator': 2},
 'duration': {'value': 0.5, 'numerator': 1, 'denominator': 2}}

In [216]:
match_tl_score.get_timestamp_for(match_first_event_id)

TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 2), quarters), source='score:clt_1ev0htiq1dt3p')

In [217]:
match_tl_score.get_timestamp_at(10)

ID,Coordinate,Type
score:clt_1ev0htiq1dt3p,10 quarters,axis
quarters,19/2 quarters,cmap
ticks,4800 ticks,cmap


In [218]:
match_tl_score.get_events_at(0)

{'score:clt_1ev0htiq1dt3p': [{'id': 'n1',
   'name': None,
   'temporal_type': 'interval',
   'event_type': None,
   'start': Fraction(0, 1),
   'end': Fraction(1, 2),
   'duration': Fraction(1, 2)}]}

In [219]:
match_event_ids = first_ids(match_tl_score)
match_tl_score.get_timestamps_for(match_event_ids)

[TimeIntervalStamp(start=Coordinate(Fraction(0, 1), quarters), end=Coordinate(Fraction(1, 2), quarters), source='score:clt_1ev0htiq1dt3p'),
 TimeIntervalStamp(start=Coordinate(Fraction(1, 2), quarters), end=Coordinate(Fraction(3, 2), quarters), source='score:clt_1ev0htiq1dt3p'),
 TimeIntervalStamp(start=Coordinate(Fraction(1, 2), quarters), end=Coordinate(Fraction(3, 4), quarters), source='score:clt_1ev0htiq1dt3p'),
 TimeIntervalStamp(start=Coordinate(Fraction(1, 2), quarters), end=Coordinate(Fraction(1, 1), quarters), source='score:clt_1ev0htiq1dt3p'),
 TimeIntervalStamp(start=Coordinate(Fraction(3, 4), quarters), end=Coordinate(Fraction(5, 4), quarters), source='score:clt_1ev0htiq1dt3p')]

In [220]:
match_tl_score.get_timestamp_table()

pyarrow.Table
score:clt_1ev0htiq1dt3p: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
raw_quarters: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
quarters_to_divs: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
score:clt_1ev0htiq1dt3p: [
  -- is_valid: all not null
  -- child 0 type: double
[0,0.5,0.75,1,1.25,...,39.75,40,40.25,40.5,41.5]
  -- child 1 type: int64
[0,1,3,1,5,...,159,40,161,81,83]
  -- child 2 type: int64
[1,2,4,1,4,...,4,1,4,2,2]]
raw_quarters: [
  -- is_valid: all not null
  -- child 0 type: double
[-0.5,0,0.25,0.5,0.75,...,39.25,39.5,39.75,40,41]
  -- child 1 type: int64
[-1,0,1,1,3,...,157,79,159,40,41]
  -- child 2 type: int64
[2,1,4,2,4,...,4,2,4,1,1]]
quarters_to_divs: [
  --

### MatchfileLoader: TimelineGroup

MatchfileLoader produces score and performance timelines that are organized
into groups by the AlignmentBundle.

In [221]:
match_bundle = match_loader.create_bundle()
match_bundle

AlignmentBundle(id='bundle_1vpmf6txmq6hd', timelines=2, groups=1)

In [222]:
match_bundle.n_groups

1

In [223]:
if match_bundle.groups:
    match_group_id = list(match_bundle.groups.keys())[0]
    match_group = match_bundle.groups[match_group_id]
    match_group

### MatchfileLoader: AlignmentBundle with MatchClaim Creation

In [224]:
match_claims = match_bundle.get_match_claims()
len(match_claims)

454

In [225]:
match_claims[0] if match_claims else "No claims"

MatchClaim(interval: score:clt_1ev0htiq1dt3p[0 quarters-1/2 quarters] <-> perf:dlt_25vknh9jrada8[0 ticks-536 ticks])

In [226]:
match_claims[0].get_matchstamp() if match_claims else "No claims"

ID,Coordinate,Type
score:clt_1ev0htiq1dt3p,0 quarters,stated
perf:dlt_25vknh9jrada8,0 ticks,stated
score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8,0 ticks,interval claim 1 (perf:dlt_25vknh9jrada8)


In [227]:
match_tl_ids = list(match_bundle.timeline_ids)[:2]
match_tl_a, match_tl_b = match_bundle.get_timelines(match_tl_ids)
print(f"Timeline A: {match_tl_a.id}, Timeline B: {match_tl_b.id}")

Timeline A: score:clt_1ev0htiq1dt3p, Timeline B: perf:dlt_25vknh9jrada8


In [228]:
# Create additional MatchClaims by pairing events
match_evts_a = list(match_tl_a.get_events())[:2]
match_evts_b = list(match_tl_b.get_events())[:2]
if match_evts_a and match_evts_b:
    match_pairs = [
        (match_evts_a[0], match_tl_ids[0], match_evts_b[0], match_tl_ids[1]),
    ]
    match_new_claims = match_bundle.create_match_claims(
        match_pairs, synchronous=True, provenance=EXAMPLE_PROVENANCE
    )
    print(f"Created {len(match_new_claims)} new MatchClaim(s)")
    match_new_claims[0] if match_new_claims else "No claims created"

Created 1 new MatchClaim(s)


Batch retrieval has two flavours. **Coordinate-batch** — "resolve THESE
coordinates" — resolves a handful of query coordinates on one timeline into one
full cross-section each. The **whole-alignment** table below instead tabulates
the entire alignment. Users hold coordinates, not `MatchClaim` objects, so the
coordinate-batch form is the recommended entry point.

A match file aligns notes, so every claim it yields is an interval claim
pairing a note's score span with its performed span. A score position can lie
in several such spans at once — the notes of a chord played slightly spread,
or a note that ends where the next begins — and each claim then contributes
its own performance candidate. The stamp lists them side by side and never
merges them; the performance belongs to the stamp's coordinates only where
all candidates agree.

In [229]:
# get_matchstamps_at(coords, timeline_id) - resolve THESE score coordinates into
# cross-group MatchStamps (raw numeric + timeline_id form), one candidate per
# interval claim containing the position.
match_query_events = list(match_tl_a.get_events())
match_query_coords = [
    match_query_events[i]["start"]["value"]
    for i in (0, len(match_query_events) // 2, -1)
]
match_stamps = match_bundle.get_matchstamps_at(match_query_coords, match_tl_ids[0])
match_stamps

[MatchStamp(score:clt_1ev0htiq1dt3p=0 quarters, perf:dlt_25vknh9jrada8=0 ticks, claim 1 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=0 ticks, claim 2 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=0 ticks),
 MatchStamp(score:clt_1ev0htiq1dt3p=91/4 quarters, claim 1 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=39243 ticks, claim 2 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=39293 ticks, claim 3 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=39158 ticks, claim 4 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=39247 ticks, claim 5 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=39255 ticks, claim 6 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vknh9jrada8=39278 ticks),
 MatchStamp(score:clt_1ev0htiq1dt3p=81/2 quarters, claim 1 score:clt_1ev0htiq1dt3p<->perf:dlt_25vknh9jrada8: perf:dlt_25vkn

In [230]:
# get_matchstamp_table(coords) - the same batch as a table, one row per query
# coordinate. A cell holds one coordinate, so only positions whose performance
# is among the stamp's present_timelines can be tabulated. Here the query
# coordinates are IdCoordinates, which carry their own timeline_id, so no
# separate timeline_id argument is needed.
from timetoalign.core import IdCoordinate

match_id_coords = [
    IdCoordinate(c, match_tl_a.unit, match_tl_ids[0])
    for c, stamp in zip(match_query_coords, match_stamps)
    if match_tl_ids[1] in stamp.present_timelines
]
match_bundle.get_matchstamp_table(match_id_coords)

pyarrow.Table
score:clt_1ev0htiq1dt3p: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
perf:dlt_25vknh9jrada8: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
score:clt_1ev0htiq1dt3p: [
  -- is_valid: all not null
  -- child 0 type: double
[0]
  -- child 1 type: int64
[0]
  -- child 2 type: int64
[1]]
perf:dlt_25vknh9jrada8: [
  -- is_valid: all not null
  -- child 0 type: double
[0]
  -- child 1 type: int64
[0]
  -- child 2 type: int64
[1]]

In [231]:
# A position with several performance candidates has no single cell value, so
# the table refuses the whole batch rather than pick one.
try:
    match_bundle.get_matchstamp_table(match_query_coords, match_tl_ids[0])
except ValueError as error:
    print(error)

get_matchstamp_table cannot materialize ambiguous interval-derived coordinates for timelines ['perf:dlt_25vknh9jrada8']


In [232]:
# get_matchstamp_table() - no arguments tabulates the WHOLE alignment (one row
# per claim), a distinct operation from the coordinate-batch above.
match_bundle.get_matchstamp_table()

pyarrow.Table
score:clt_1ev0htiq1dt3p: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
perf:dlt_25vknh9jrada8: struct<value: double, numerator: int64, denominator: int64>
  child 0, value: double
  child 1, numerator: int64
  child 2, denominator: int64
----
score:clt_1ev0htiq1dt3p: [
  -- is_valid: all not null
  -- child 0 type: double
[0,0.5,0.5,1,0.5,...,41.5,40.5,41.5,0,0.5]
  -- child 1 type: int64
[0,1,1,1,1,...,83,81,83,0,1]
  -- child 2 type: int64
[1,2,2,1,2,...,2,2,2,1,2]]
perf:dlt_25vknh9jrada8: [
  -- is_valid: all not null
  -- child 0 type: double
[0,536,526,1319,555,...,77422,73798,77497,0,536]
  -- child 1 type: int64
[0,536,526,1319,555,...,77422,73798,77497,0,536]
  -- child 2 type: int64
[1,1,1,1,1,...,1,1,1,1,1]]

***
## Summary

This notebook demonstrates the unified API across all TimeToAlign! loaders:

| API Layer | Key Methods |
|-----------|-------------|
| **Loader** | `load()`, `from_file()`, `store`, `create_timeline()`, `create_timelines()` |
| **Timeline** | `get_events()`, `get_event()`, `get_timestamp_at()`, |
| | `get_timestamp_for()`, `get_timestamps_for()`, `get_timestamp_table()` |
| **TimelineGroup** | `get_timeline()`, `get_events()`, `get_timestamp_at()`, |
| | `get_timestamp_for()`, `get_timestamps_for()`, `get_timestamp_table()` |
| **AlignmentBundle** | `get_timelines()`, `get_match_claims()`, `create_match_claims()` |
| | `get_matchstamp_at()`, `get_matchstamps_at()`, `get_matchstamp_table()` |

The Timeline and TimelineGroup rows are the same four names because retrieval
is one grid: `_at` for a position, `_for` for a key, the plural for many of
them, and `get_*_table()` for the tabular form of the same query. The bundle
row is that grid again, spelled `matchstamp` because a row there crosses
timelines rather than staying inside one.